<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L04-lora-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L04-lora-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L04-lora-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L04-lora-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T04-L04 · LoRA from scratch: low-rank adapters on the nanolm, and the rank at which they stop helping

**You will build:** a LoRA linear layer with a hand-derived backward pass, wire it into the
numpy transformer you met in `T03-L03-nanolm-cpu-slice`, adapt that model to a vocabulary it has
never seen at several ranks, and measure what the adapter costs, what it buys, and the rank at
which a bigger one stops buying anything.

**Time:** ~120 minutes · **Runs on:** a laptop CPU, no GPU, no download, far under 8 GiB ·
**Run all:** about five minutes on a Colab CPU, less on a recent laptop

By the end you will be able to:
1. Implement a LoRA linear layer (initialisation, forward, hand-derived backward and merge) and
   verify its gradients against central finite differences.
2. Measure the trainable parameters and optimiser-state bytes of rank-r adapters against full
   fine-tuning in exact counts, and compute the rank beyond which an adapter has more
   parameters than the matrix it adapts.
3. Fine-tune one pretrained numpy transformer to a new vocabulary at six ranks and three
   learning rates, and measure held-out loss for every pair against full fine-tuning.
4. Find the rank beyond which a larger adapter stops lowering the validation loss under a fixed
   protocol, and measure how far that rank moves when the learning rate or the scale rule changes.
5. Measure how the scaling rule (`alpha / r` or `alpha / sqrt(r)`) changes the first steps of
   training at different ranks, and how many directions the full fine-tuning update uses.

**No PyTorch, no PEFT.** The model is the numpy transformer block of `T03-L03`, copied in and
given a place to plug an adapter into. It pretrains its own tiny checkpoint below, because a
lesson cannot load another lesson's files. Everything that `peft.LoraConfig`
hides (the initialisation, the `alpha / r` scale, the merge) is a function you write.

**About the `if __name__ == "__main__":` guards.** They let the autograder import this file
without running the fine-tunes. A Jupyter kernel sets `__name__` to `"__main__"`, so every
guarded cell still runs when you execute the notebook top to bottom.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T03-L03`, `T04-L02`

**This lesson assumes you can already:**

- multiply matrices with @ in numpy and predict the shape of the result, including for a batch of matrices
- differentiate a composition of functions with the chain rule, and say what the gradient of a matrix product is with respect to each factor
- say what one step of Adam does to a parameter, and why a parameter with a zero gradient never moves
- read a held-out loss curve and say why a validation set and a test set are kept apart (T03-L03 trains a language model this way)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) · Hu et al. (Microsoft), arXiv · *free* — section 4 is the whole method (the low-rank update, the zero initialisation of B, the alpha / r scale) and section 7.2 is the rank experiment this lesson repeats in miniature
- [LoRA Without Regret](https://thinkingmachines.ai/blog/lora/) · John Schulman and others, Thinking Machines Lab · *free* — rank, learning-rate and adapter-placement sweeps at the scale of real models, and the claims about learning rates and scaling that the first-steps and sweep sections test on a small model
- [LoRA (PEFT developer guide)](https://huggingface.co/docs/peft/developer_guides/lora) · Hugging Face · *free* — how LoRA is configured in the library most people will use, including merging and unmerging an adapter, which is what the optional cross-check at the end of the lesson would compare against
- [CS231n: Backpropagation, Intuitions](https://cs231n.github.io/optimization-2/) · Stanford CS231n (Karpathy and Li) · *free* — the gradient of a matrix product with respect to each factor, which exercise 3 applies twice to get the gradients of A, B and the input

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed by the code itself.
import contextlib
import os
import signal
import sys
import threading
import time
from typing import Callable, NamedTuple

import numpy as np

print("python", sys.version.split()[0])
print("numpy ", np.__version__)

# Every check and every demo below runs through `_try`, so pressing Run all before you have
# written a line reaches the progress board at the foot of the notebook instead of stopping at
# the first stub.
# The nine exercises, in the order the notebook meets them: (label, function, where).
_EXERCISES = [("exercise 1", "init_adapter", "section 4"),
              ("exercise 2", "lora_forward", "section 5"),
              ("exercise 3", "lora_backward", "section 5"),
              ("exercise 4", "merge_adapter", "section 6"),
              ("exercise 5", "lora_scale", "section 7"),
              ("exercise 6", "adapter_footprint", "section 8"),
              ("exercise 7", "break_even_rank", "section 8"),
              ("exercise 8", "find_knee", "section 9"),
              ("exercise 9", "effective_rank", "section 11")]
_FUNCTION = {label: function for label, function, _ in _EXERCISES}
_FOR_TRAINING = ("exercise 1", "exercise 2", "exercise 3", "exercise 5")   # what a LoRA fine-tune calls
_STATUS: dict = {}       # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}   # label -> the stub whose NotImplementedError stopped it


def _unfinished(exc: BaseException) -> str:
    """The function that raised NotImplementedError: the innermost frame of the traceback.

    A fine-tune calls the adapter functions of exercises 1 to 3, so an unfilled `lora_forward`
    stops the training demos too, and naming the stub that actually raised says which one to go
    back to.
    """
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (`lora_backward`)"; several exercises -> "exercises 1, 2 and 3";
    a demo that has not run -> its own name."""
    if all(label in _FUNCTION for label in labels) and len(labels) > 1:
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} (`{_FUNCTION[label]}`)" if label in _FUNCTION else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so, by name. A wrong answer prints the
    check's own message, which names the likely mistake, and the notebook carries on, so one
    broken exercise never hides the feedback on the others. A demo names the exercises it
    `needs`: until each has passed its check, the demo says which one it is waiting for and
    skips, so it never runs on an answer a check has just rejected. Nothing is swallowed:
    every outcome is recorded for the progress board, and a script run exits non-zero there if
    any check came back wrong. A check still running after `_CHECK_SECONDS` (a demo:
    `_DEMO_SECONDS`) is stopped and counts as failed, so a loop that never ends cannot hang the
    notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        limit, why = (_CHECK_SECONDS, _CHECK_WHY) if label in _FUNCTION else (_DEMO_SECONDS, _DEMO_WHY)
        with _time_limit(limit, label, why):
            check()
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, function in _FUNCTION.items() if function == stub]
        if owner and owner[0] != label:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises "
                  "NotImplementedError.")
        else:
            todo = f"`{stub}`" if stub else "a stub"
            print(f"{label}: not implemented yet, {todo} still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


# A check that never returns would freeze the notebook, and a loop that appends as it spins can
# eat gigabytes in seconds. Every exercise check runs under a time limit far above what a right
# answer needs (each takes milliseconds); a demo that trains gets a longer one.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 180
_CHECK_WHY = "a loop that never reaches its stopping condition never ends"
_DEMO_WHY = ("this demo trains many models, and a slow or shared CPU can need longer than its limit: run "
             "the cell again on an idle machine, or raise _DEMO_SECONDS above")


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str,
                why: str = "a loop that never reaches its stopping condition never ends",
                memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that was already running (the grader's own
    time limit, or an enclosing guard) keeps its deadline if it is due first and is put back,
    with the time it had left, afterwards. A correct answer finishes far inside the limit, and
    nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = f"{what} was still running after {seconds:g} s: {why}"
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _peak_bytes(call: Callable[[], object]) -> int:
    """The most memory, in bytes, that the call had allocated at one moment while it ran. Python's
    `tracemalloc` sees numpy's array allocations, so this is how the checks tell a layer that never
    builds a `d_in x d_out` array from one that does. Whatever the call needs must exist before it."""
    import tracemalloc
    already = tracemalloc.is_tracing()
    if not already:
        tracemalloc.start()
    try:
        tracemalloc.reset_peak()
        before = tracemalloc.get_traced_memory()[0]
        call()
        return tracemalloc.get_traced_memory()[1] - before
    finally:
        if not already:
            tracemalloc.stop()


# Everything runs in float64: the finite-difference checks below are meaningless in float32.
DTYPE = np.float64

## 1. The model: the T03-L03 nanolm, with a place for an adapter

The numpy transformer block of `T03-L03-nanolm-cpu-slice` is copied in below: token and
position embeddings, one causal attention head (`Wq`, `Wk`, `Wv`), an output projection `Wo` on a
residual connection, and a linear read-out `W1` with a bias. Its hand-written backward pass is
given and is proved against finite differences in the next cell. The vocabulary is the 26
lowercase letters, a space, a full stop and a newline: 29 characters (the vocabulary and the size
of the initial weights are this lesson's own, not `T03-L03`'s). Two changes to the code exist so
that an adapter has somewhere to go:

- **Each of the five weight matrices is applied through one helper.** When an adapter is attached
  to a matrix, the helper calls *your* `lora_forward` and `lora_backward` (exercises 2 and 3)
  instead of a plain `x @ W`. Embeddings and the bias are never adapted: LoRA leaves them frozen.
- **The backward pass can skip the weight gradients.** A frozen matrix gets no gradient at all,
  which is the memory saving LoRA is for; only its input gradient flows on.

Run the cell. It defines the model and prints how many parameters it has and where they live.

In [ ]:
CHARS = "abcdefghijklmnopqrstuvwxyz .\n"
VOCAB = len(CHARS)
STOI = {c: i for i, c in enumerate(CHARS)}
D_MODEL = 48         # width of the residual stream
CONTEXT = 32         # positions the model has an embedding for
INIT_SCALE = 0.1     # standard deviation of the initial weights
LAYERS = ("Wq", "Wk", "Wv", "Wo", "W1")   # the five matrices an adapter can sit on


def softmax(x: np.ndarray) -> np.ndarray:
    """Softmax over the last axis, shifted by the row maximum so it cannot overflow."""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def init_params(seed: int = 0, scale: float = INIT_SCALE) -> dict:
    """Fresh parameters, float64, from a seeded generator."""
    rng = np.random.default_rng(seed)
    params = {"E": rng.normal(0, scale, (VOCAB, D_MODEL)),
              "P": rng.normal(0, scale, (CONTEXT, D_MODEL))}
    for name in ("Wq", "Wk", "Wv", "Wo"):
        params[name] = rng.normal(0, scale, (D_MODEL, D_MODEL))
    params["W1"] = rng.normal(0, scale, (D_MODEL, VOCAB))
    params["b1"] = np.zeros(VOCAB, dtype=DTYPE)
    return params


def count_params(tree: dict) -> int:
    """Number of scalars in a dict of arrays."""
    return int(sum(v.size for v in tree.values()))


def _linear(x, params, adapters, scale, name):
    """One weight matrix: `x @ W`, plus the adapter on it when there is one. Returns (y, cache)."""
    if name in adapters:
        return lora_forward(x, params[name], adapters[name], scale)
    return x @ params[name], None


def _linear_back(dy, x, params, adapters, name, cache, grads, adapter_grads, train_base):
    """Backward of `_linear`: returns the gradient with respect to the input, and records the
    weight gradient (full fine-tuning) or the adapter gradient (LoRA, through YOUR backward)."""
    if name in adapters:
        dx, adapter_grads[name] = lora_backward(dy, cache)
        return dx
    if train_base:
        grads[name] = x.reshape(-1, x.shape[-1]).T @ dy.reshape(-1, dy.shape[-1])
    return dy @ params[name].T


def forward(params, adapters, scale, ids):
    """ids: (B, T) ints. Returns (logits (B, T, V), cache). `adapters` maps a matrix name to its
    adapter, or is empty for the plain model; `scale` is the adapter scale (alpha / r or similar)."""
    t = ids.shape[1]
    x = params["E"][ids] + params["P"][None, :t]
    cache = {"ids": ids, "x": x}
    q, cache["Wq"] = _linear(x, params, adapters, scale, "Wq")
    k, cache["Wk"] = _linear(x, params, adapters, scale, "Wk")
    v, cache["Wv"] = _linear(x, params, adapters, scale, "Wv")
    sc = 1.0 / np.sqrt(D_MODEL)
    mask = np.tril(np.ones((t, t), dtype=bool))
    a = softmax(np.where(mask, (q @ k.transpose(0, 2, 1)) * sc, -np.inf))
    ctx = a @ v
    o, cache["Wo"] = _linear(ctx, params, adapters, scale, "Wo")
    h = x + o
    logits, cache["W1"] = _linear(h, params, adapters, scale, "W1")
    logits = logits + params["b1"]
    cache.update(q=q, k=k, v=v, a=a, ctx=ctx, h=h, sc=sc, mask=mask)
    return logits, cache


def mean_nll(logits, targets) -> float:
    """Mean negative log-likelihood in nats."""
    m = logits.max(axis=-1, keepdims=True)
    lse = m[..., 0] + np.log(np.exp(logits - m).sum(axis=-1))
    return float((lse - np.take_along_axis(logits, targets[..., None], axis=-1)[..., 0]).mean())


def backward(params, adapters, scale, cache, logits, targets, train_base):
    """Gradients of `mean_nll`. Returns (grads, adapter_grads): `grads` holds a gradient for every
    parameter when `train_base` is True (full fine-tuning, no adapters), and is empty when it is
    False; `adapter_grads[name]` is what `lora_backward` returned for that matrix."""
    assert not (adapters and train_base), "a model is either fully fine-tuned or adapted, not both"
    d = D_MODEL
    grads, adapter_grads = {}, {}
    d_logits = softmax(logits)
    chosen = np.take_along_axis(d_logits, targets[..., None], axis=-1)
    np.put_along_axis(d_logits, targets[..., None], chosen - 1.0, axis=-1)
    d_logits = d_logits / targets.size
    if train_base:
        grads["b1"] = d_logits.sum(axis=(0, 1))
    d_h = _linear_back(d_logits, cache["h"], params, adapters, "W1", cache["W1"], grads,
                       adapter_grads, train_base)
    d_ctx = _linear_back(d_h, cache["ctx"], params, adapters, "Wo", cache["Wo"], grads,
                         adapter_grads, train_base)
    a, q, k, v = cache["a"], cache["q"], cache["k"], cache["v"]
    d_a = d_ctx @ v.transpose(0, 2, 1)
    d_v = a.transpose(0, 2, 1) @ d_ctx
    d_scores = a * (d_a - (d_a * a).sum(axis=-1, keepdims=True))
    d_scores = np.where(cache["mask"], d_scores, 0.0) * cache["sc"]
    d_q, d_k = d_scores @ k, d_scores.transpose(0, 2, 1) @ q
    x = cache["x"]
    d_x = d_h.copy()                       # the residual path: h = x + o
    for name, d_out in (("Wq", d_q), ("Wk", d_k), ("Wv", d_v)):
        d_x += _linear_back(d_out, x, params, adapters, name, cache[name], grads, adapter_grads,
                            train_base)
    if train_base:
        grads["P"] = np.zeros_like(params["P"])
        grads["P"][:x.shape[1]] = d_x.sum(axis=0)
        grads["E"] = np.zeros_like(params["E"])
        np.add.at(grads["E"], cache["ids"], d_x)
    return grads, adapter_grads


def adam_init(tree: dict) -> dict:
    """Adam's state for a dict of arrays: first moment, second moment, step count."""
    return {k: [np.zeros_like(v), np.zeros_like(v), 0] for k, v in tree.items()}


def adam_step(tree, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8) -> None:
    """One bias-corrected Adam update of every array in `tree`, in place."""
    for k, p in tree.items():
        m, v, _ = state[k]
        state[k][2] += 1
        t = state[k][2]
        m *= beta1
        m += (1 - beta1) * grads[k]
        v *= beta2
        v += (1 - beta2) * grads[k] ** 2
        p -= lr * (m / (1 - beta1 ** t)) / (np.sqrt(v / (1 - beta2 ** t)) + eps)


def base_gradient_check(seed: int = 900, scale: float = 0.2, probes: int = 6) -> float:
    """Finite-difference every parameter of the plain model. Returns the worst error as a fraction
    of the tolerance (anything above 1.0 is a real disagreement)."""
    rng = np.random.default_rng(seed)
    params = init_params(seed, scale)
    params["b1"] = rng.normal(0, 0.02, VOCAB)
    ids = rng.integers(0, VOCAB, (3, 9))
    targets = rng.integers(0, VOCAB, (3, 9))

    def loss_of() -> float:
        return mean_nll(forward(params, {}, 1.0, ids)[0], targets)

    logits, cache = forward(params, {}, 1.0, ids)
    grads, _ = backward(params, {}, 1.0, cache, logits, targets, True)
    worst = 0.0
    for name, array in params.items():
        flat = array.reshape(-1)
        for q in rng.integers(0, flat.size, size=probes):
            old = flat[q]
            flat[q] = old + 1e-5
            up = loss_of()
            flat[q] = old - 1e-5
            down = loss_of()
            flat[q] = old
            numeric, analytic = (up - down) / 2e-5, float(grads[name].reshape(-1)[q])
            tolerance = 1e-9 + 1e-5 * max(abs(numeric), abs(analytic))
            worst = max(worst, abs(numeric - analytic) / tolerance)
    return worst


if __name__ == "__main__":
    _worst = base_gradient_check()
    _params = init_params()
    print(f"vocabulary: {VOCAB} characters; parameters: {count_params(_params)} in {len(_params)} arrays; "
          f"the five matrices an adapter can sit on hold {sum(_params[n].size for n in LAYERS)} of them")
    # The worst error itself is rounding noise, which changes with the BLAS library underneath
    # numpy, so the cell prints the verdict, not the ratio.
    print(f"gradient check of the plain model, every parameter block: every probed derivative "
          f"within its tolerance: {_worst <= 1.0}")
    assert _worst <= 1.0, "the copied backward pass disagrees with finite differences"

## 2. A base model, and a new domain

LoRA adapts a model that already exists, so the lesson needs one. Two tiny synthetic "domains",
both made of random words from a fixed word list, one word after another:

- **Domain A, the base's world:** 40 everyday words that between them use every letter. The base
  model pretrains on it for a fixed number of steps, which the cell prints.
- **Domain B, the new domain:** 20 words from a different vocabulary. The base has never seen
  one of them, and the task is the one a real adapter gets: keep what the model knows about
  letters and attention, learn a new lexicon.

Every corpus is generated from a fixed seed, so nothing is downloaded. Domain B has a training
set, a **validation** set (where learning rates are chosen) and a **test** set (reported once,
never tuned on). A word of caution from the original LoRA paper (a footnote in its section 7.2;
`claims.yaml`): a small rank may not work for every task, and its example is a downstream task in a
different language from the pre-training, where retraining the whole model could do better than
LoRA with a small rank. A vocabulary the base has never seen is that case in miniature, which is
why the sweep below goes up to a rank that already costs more than the matrices it adapts. Run the cell.

In [ ]:
WORDS_A = ["the", "quick", "brown", "fox", "jumps", "over", "lazy", "dog", "a", "cat", "sat", "on",
           "mat", "red", "big", "sun", "and", "moon", "fell", "up", "down", "in", "one", "two",
           "zebra", "wax", "vex", "jazz", "fix", "box", "gym", "kiwi", "quartz", "job", "fly",
           "by", "him", "her", "we", "you"]
WORDS_B = ["bird", "tree", "wind", "rain", "gold", "blue", "fire", "leaf", "road", "rock", "salt",
           "wave", "west", "east", "cold", "fast", "slow", "sing", "star", "dusk"]
PRETRAIN_STEPS = 800
PRETRAIN_LR = 3e-3
BATCH = 16
WINDOW = 32          # characters per training sequence: one position id for every embedding


def make_corpus(words: list, seed: int, lines: int = 300, per_line: int = 30) -> np.ndarray:
    """`lines` lines of `per_line` random words, a full stop and a newline, as character ids."""
    rng = np.random.default_rng(seed)
    text = "".join(" ".join(rng.choice(words, per_line)) + ".\n" for _ in range(lines))
    return np.array([STOI[c] for c in text])


def decode(ids) -> str:
    return "".join(CHARS[int(i)] for i in ids)


def windows(data: np.ndarray, rng, n: int):
    """n random windows of WINDOW characters and their next-character targets."""
    starts = rng.integers(0, len(data) - WINDOW - 1, n)
    return (np.stack([data[s:s + WINDOW] for s in starts]),
            np.stack([data[s + 1:s + WINDOW + 1] for s in starts]))


def evaluate(params, adapters, scale, data, batches: int = 40, seed: int = 5) -> float:
    """Mean loss in nats per character over the same 640 windows every time it is called."""
    rng = np.random.default_rng(seed)
    total = 0.0
    for _ in range(batches):
        x, y = windows(data, rng, 16)
        total += mean_nll(forward(params, adapters, scale, x)[0], y)
    return total / batches


def pretrain_base(corpus: np.ndarray, seed: int = 0, steps: int = PRETRAIN_STEPS):
    """The base checkpoint: Adam on random windows of domain A. Returns (params, first, last)."""
    rng = np.random.default_rng(seed)
    params = init_params(seed)
    state = adam_init(params)
    losses = []
    for _ in range(steps):
        x, y = windows(corpus, rng, BATCH)
        logits, cache = forward(params, {}, 1.0, x)
        losses.append(mean_nll(logits, y))
        grads, _ = backward(params, {}, 1.0, cache, logits, y, True)
        adam_step(params, grads, state, PRETRAIN_LR)
    return params, float(np.mean(losses[:10])), float(np.mean(losses[-10:]))


if __name__ == "__main__":
    A_TRAIN, A_TEST = make_corpus(WORDS_A, 10, 120), make_corpus(WORDS_A, 11, 120)
    B_TRAIN, B_VAL, B_TEST = (make_corpus(WORDS_B, s) for s in (20, 21, 22))
    BASE, _first, _last = pretrain_base(A_TRAIN)
    BASE0 = {k: v.copy() for k, v in BASE.items()}     # a copy, to prove later that LoRA never touches it
    print(f"domain A: {len(WORDS_A)} words using {len(set(''.join(WORDS_A)))} different letters, "
          f"{len(A_TRAIN)} training characters; domain B: {len(WORDS_B)} words using "
          f"{len(set(''.join(WORDS_B)))} letters, {len(B_TRAIN)} training, {len(B_VAL)} validation, "
          f"{len(B_TEST)} test characters; words in both lists: {len(set(WORDS_A) & set(WORDS_B))}")
    print(f"pretraining on A: loss {_first:.3f} -> {_last:.3f} nats per character over "
          f"{PRETRAIN_STEPS} steps (guessing uniformly scores {np.log(VOCAB):.3f})")
    print("a line of domain A:", repr(decode(A_TRAIN[:60])))
    print("a line of domain B:", repr(decode(B_TRAIN[:60])))
    _base_a, _base_b = evaluate(BASE, {}, 1.0, A_TEST), evaluate(BASE, {}, 1.0, B_TEST)
    print(f"the base model on held-out A: {_base_a:.3f} nats; on held-out B: {_base_b:.3f} nats "
          f"(a model that is {_base_b / _base_a:.1f} times worse on B than on A)")

## 3. The baseline: full fine-tuning, and what it costs

Before any adapter, the reference everyone compares against: **full fine-tuning**, which updates
every parameter. The helper `finetune` below is given. It runs Adam for a fixed number of steps
with a learning-rate schedule (a linear **warm-up** over the first `WARMUP` steps, then a linear
decay to zero), and counts two things from the arrays themselves rather than from a formula:

- **trainable parameters**: how many scalars the optimiser updates;
- **state bytes**: the bytes held by those parameters, their gradients, and Adam's two moment
  arrays. This is the memory a training run needs *on top of* the frozen model, and it is the
  number LoRA exists to shrink.

For an adapter run (the `rank` argument), `finetune` calls your exercises 1, 2, 3 and 5, so it
only works once they pass. Full fine-tuning needs none of them, so the cell below runs now. The
learning rates are a grid, `LRS`, and the best one is chosen on the **validation** set; the
**test** loss is what is reported. The cell after it explains why the grid stops where it does.

In [ ]:
ALPHA = 16.0                       # the adapter scale numerator: scale = alpha / rank
# PEFT's LoraConfig defaults as of 14 October 2026 (quoted in claims.yaml): rank 8, lora_alpha 8,
# lora_dropout 0.0. This lesson picks its own alpha and ranks; the defaults are printed for comparison.
PEFT_DEFAULTS = {"r": 8, "lora_alpha": 8, "lora_dropout": 0.0}
LRS = (3e-3, 1e-2, 3e-2)           # the learning-rate grid, for full fine-tuning and for every rank
STEPS = 600                        # optimiser steps per run
WARMUP = 60                        # steps over which the learning rate rises linearly from 0 to its full value
SEEDS = (2, 3)                     # batch order and, for adapters, initialisation
RANKS = (1, 2, 4, 8, 16, 32)
STATE_COPIES = 4                   # per trainable scalar: the value, its gradient, Adam's m and v


class Run(NamedTuple):
    params: dict         # the weights the model runs with: the base itself (untouched) for an adapter run
    adapters: dict       # {matrix name: {"A": ..., "B": ...}}, empty for full fine-tuning
    scale: float         # what the adapter output is multiplied by
    trainable: int       # scalars the optimiser updated
    state_bytes: int     # bytes of trainables + gradients + Adam moments, summed over the real arrays
    watch: dict          # {step: probe loss} for the steps asked for with `watch`


def finetune(base, data, *, lr, steps=STEPS, seed=2, rank=None, alpha=ALPHA, rule="lora",
             layers=LAYERS, decay=True, warmup=WARMUP, watch=(), probe=None) -> Run:
    """Adam for `steps` steps on random windows of `data`.

    The learning rate rises linearly from 0 over the first `warmup` steps and then falls linearly
    to 0 at the last step; `decay=False` holds it constant at `lr` and `warmup=0` skips the rise.

    rank=None: full fine-tuning of a copy of `base` (every parameter trains).
    rank=r:    LoRA. `base` is not copied and not modified; an adapter of rank r is attached to each
               matrix in `layers` with YOUR `init_adapter`, scaled by YOUR `lora_scale(alpha, r, rule)`,
               and only the adapters train.
    watch:     steps at which to record the loss on the fixed batch `probe` (0 = before any step).
    """
    rng = np.random.default_rng(seed)
    if rank is None:
        params, adapters, scale = {k: v.copy() for k, v in base.items()}, {}, 1.0
        tree = params
    else:
        params = base
        adapters = {name: init_adapter(base[name].shape[0], base[name].shape[1], rank,
                                       seed=1000 * seed + i)
                    for i, name in enumerate(layers)}
        scale = lora_scale(alpha, rank, rule)
        tree = {f"{name}.{part}": adapters[name][part] for name in adapters for part in ("A", "B")}
    state = adam_init(tree)
    seen, last_grads = {}, tree
    for step in range(steps + 1):
        if step in watch:
            seen[step] = mean_nll(forward(params, adapters, scale, probe[0])[0], probe[1])
        if step == steps:
            break
        x, y = windows(data, rng, BATCH)
        logits, cache = forward(params, adapters, scale, x)
        grads, adapter_grads = backward(params, adapters, scale, cache, logits, y, rank is None)
        if rank is not None:
            grads = {f"{name}.{part}": adapter_grads[name][part] for name in adapter_grads
                     for part in ("A", "B")}
        last_grads = grads
        rate = lr * min(1.0, (step + 1) / warmup) if warmup else lr
        adam_step(tree, grads, state, rate * (1 - step / steps) if decay else rate)
    held = sum(a.nbytes for a in tree.values()) + sum(g.nbytes for g in last_grads.values()) \
        + sum(m.nbytes + v.nbytes for m, v, _ in state.values())
    return Run(params, adapters, scale, count_params(tree), int(held), seen)


def score(run: Run, data) -> float:
    """Held-out loss of a finished run on `data`."""
    return evaluate(run.params, run.adapters, run.scale, data)


if __name__ == "__main__":
    FULL = {}
    for _lr in LRS:
        for _seed in SEEDS:
            _run = finetune(BASE, B_TRAIN, lr=_lr, seed=_seed)
            FULL[_lr, _seed] = (_run, score(_run, B_VAL), score(_run, B_TEST), score(_run, A_TEST))
    print(f"full fine-tuning: {STEPS} steps, batch {BATCH}, mean of {len(SEEDS)} seeds")
    print(f"{'lr':>8} {'val':>8} {'test':>8} {'held-out A':>11}")
    for _lr in LRS:
        _rows = [FULL[_lr, s] for s in SEEDS]
        print(f"{_lr:>8.0e} {np.mean([r[1] for r in _rows]):>8.3f} {np.mean([r[2] for r in _rows]):>8.3f} "
              f"{np.mean([r[3] for r in _rows]):>11.3f}")
    FULL_LR = min(LRS, key=lambda lr: np.mean([FULL[lr, s][1] for s in SEEDS]))
    _any = FULL[FULL_LR, SEEDS[0]][0]
    print(f"best learning rate on validation: {FULL_LR:.0e}"
          + ("  (the largest on the grid: the optimum may lie beyond it)" if FULL_LR == max(LRS)
             else "  (the smallest on the grid: the optimum may lie below it)" if FULL_LR == min(LRS)
             else ""))
    print(f"it trains {_any.trainable} parameters and holds {_any.state_bytes} bytes of training "
          f"state ({STATE_COPIES} arrays of 8-byte floats per trainable scalar, measured from the arrays)")
    print(f"(the base model before fine-tuning scored {_base_a:.3f} nats on held-out A and "
          f"{_base_b:.3f} on held-out B)")

### Why the grid stops at its largest rate: a result must survive a rounding error

A learning-rate grid should reach the best rate, but a rate at which a run is not reproducible cannot
be reported. Computers differ in the last bits of a matrix product (a different maths library, a
different processor, the same library with a different number of threads), so the same program gives
weights that differ in their last few bits. At a high enough learning rate that tiny difference grows,
step after step, until the final loss is different. A number like that describes the machine, not the
model. So this grid stops at the highest rate that passes the test below, and the price is that the
best rate of a configuration may lie above the grid. The sweep says so whenever its best rate sits on
the grid's edge, and what it reports then is the best reproducible rate, not necessarily the best rate.

The cell below tests it directly. It fine-tunes the model twice from the same base, once as it is
and once with every weight nudged by one unit in the last place (`np.nextafter`, the next float up,
a smaller change than two maths libraries make), and asks whether the validation loss moved by more
than `1e-7`. The first row is the largest rate on the grid with the schedule the lesson uses, the
second removes the warm-up, and the third uses `PROBE_LR`, a rate above the grid.

In [ ]:
NUDGE_TOLERANCE = 1e-7     # a validation loss that moves more than this under a 1-ulp nudge is not reproducible
PROBE_LR = 5e-2            # a learning rate above the grid


def nudged(base: dict) -> dict:
    """A copy of `base` in which every non-zero weight is the next float up: one unit in the last place."""
    return {k: (np.nextafter(v, np.inf) if np.any(v) else v.copy()) for k, v in base.items()}


def _demo_reproducible() -> None:
    other = nudged(BASE)
    print(f"full fine-tuning, seed {SEEDS[0]}, {STEPS} steps; does the final validation loss move by more than "
          f"{NUDGE_TOLERANCE:g} when every weight of the base is nudged by one unit in the last place?")
    print(f"{'lr':>8}  {'schedule':<18}{'moves':>7}")
    verdicts = {}
    for lr, warmup in ((LRS[-1], WARMUP), (LRS[-1], 0), (PROBE_LR, WARMUP)):
        a = score(finetune(BASE, B_TRAIN, lr=lr, seed=SEEDS[0], warmup=warmup), B_VAL)
        b = score(finetune(other, B_TRAIN, lr=lr, seed=SEEDS[0], warmup=warmup), B_VAL)
        verdicts[lr, warmup] = abs(a - b) > NUDGE_TOLERANCE
        print(f"{lr:>8.0e}  {'warm-up and decay' if warmup else 'decay only':<18}{str(verdicts[lr, warmup]):>7}")
    print("the largest rate on the grid is reproducible with the warm-up: "
          f"{not verdicts[LRS[-1], WARMUP]}; without it: {not verdicts[LRS[-1], 0]}; above the grid, with it: "
          f"{not verdicts[PROBE_LR, WARMUP]}")


if __name__ == "__main__":
    _demo_reproducible()

## 4. Exercise 1 · `init_adapter(d_in, d_out, rank, seed)`

LoRA freezes a weight matrix `W` (shape `d_in x d_out`, applied as `x @ W`) and learns a
low-rank change to it. The change is a product of two thin matrices, `A` (`d_in x rank`) and
`B` (`rank x d_out`), so `delta_W = scale * A @ B` has rank at most `rank`, and training touches
`rank * (d_in + d_out)` numbers instead of `d_in * d_out`.

The *initialisation* is the first thing LoRA gets right and the first thing a re-implementation
gets wrong. The paper draws `A` at random and sets `B` to zero, so `A @ B` is zero at the start:
the adapted model begins as exactly the base model. This exercise writes that rule. The sources
for it, and for the `alpha / r` scale of exercise 5, are quoted in `claims.yaml`.

**How big should `A` be?** The paper says only "a random Gaussian initialization" and gives no standard
deviation. The authors' reference code and PEFT both draw `A` uniformly on +-1/sqrt(d_in) (as of 15
October 2026; `claims.yaml` has the source lines), whose standard deviation is 1/sqrt(3 d_in). "LoRA
Without Regret" shows that with Adam the initialisation scale of `A`, the scale `alpha` and the two
learning rates are partly redundant (of those four hyperparameters, two are redundant), so a different
init scale acts like a different learning rate or `alpha`, not like a different method. This
exercise's reference draws a Gaussian with standard deviation 1/sqrt(d_in), so that `x @ A` has entries of
about the size of `x`, and the check accepts any standard deviation from 0.55 to 1.05 times that: the
library's draw passes. Much larger or smaller scales break the size of `x @ A`. The numbers your
later runs print depend on the scale you choose.

<details><summary>💡 Hint 1 — what to think about</summary>

Which of the two matrices must be zero, and why not both? Write down what `A @ B` is when
`B = 0`, and then what the gradient of the loss with respect to `A` is, if `delta_W` only enters
the output through `B`. The cell after exercise 3 measures what two wrong initialisations do.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Make one seeded generator from `seed`. Draw `A` with independent entries of mean zero and a standard
deviation of the order of `1 / sqrt(d_in)`, so that `x @ A` has entries of about the size of `x`
itself (a Gaussian is the simplest way to get exactly that; the library's uniform draw is accepted
too). `B` is an array of zeros. Both are float64 and neither shares memory with the other.

</details>

In [ ]:
def init_adapter(d_in: int, d_out: int, rank: int, seed: int = 0) -> dict:
    """A fresh LoRA adapter for a `d_in x d_out` matrix: {"A": (d_in, rank), "B": (rank, d_out)}.

    Rules:
        * A has independent entries with mean 0 and a standard deviation between 0.55 / sqrt(d_in)
          and 1.05 / sqrt(d_in). The reference draws a Gaussian with exactly 1 / sqrt(d_in); a
          uniform draw on +-sqrt(3 / d_in) has the same standard deviation, and PEFT's own default
          (a uniform draw on +-1 / sqrt(d_in), standard deviation 1 / sqrt(3 d_in), which is 1 / sqrt(3)
          times the reference's) is inside the accepted range too;
        * B is all zeros;
        * both are float64, and the same seed gives the same adapter, a different seed a different A.

    Example:
        >>> ad = init_adapter(48, 29, 4, seed=1)
        >>> ad["A"].shape, ad["B"].shape
        ((48, 4), (4, 29))
        >>> float(abs(ad["B"]).max())
        0.0

    Returns:
        A dict with the two arrays.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement init_adapter")


def _check_init_adapter() -> None:
    ad = init_adapter(48, 29, 4, seed=1)
    assert isinstance(ad, dict) and set(ad) == {"A", "B"}, (
        f"return a dict with the keys 'A' and 'B', got {type(ad).__name__}")
    assert ad["A"].shape == (48, 4) and ad["B"].shape == (4, 29), (
        f"A must be (d_in, rank) = (48, 4) and B (rank, d_out) = (4, 29); got {ad['A'].shape} "
        f"and {ad['B'].shape}")
    assert not np.any(ad["B"]), (
        "B must start as exact zeros: it is what makes the adapted model equal the base model "
        "before the first step")
    assert np.any(ad["A"]) and np.abs(ad["A"]).max() > 0.01, (
        "A must start random. If A and B were both zero, neither would ever receive a gradient: "
        "the gradient of A passes through B, and the gradient of B passes through A")
    big = init_adapter(400, 20, 50, seed=3)["A"]
    ratio = float(big.std() * np.sqrt(400))
    assert 0.55 < ratio < 1.05, (
        f"the entries of A have standard deviation {big.std():.4f}, which is {ratio:.2f} times "
        "1/sqrt(d_in); the accepted range is 0.55 to 1.05 times that (a Gaussian of exactly 1/sqrt(d_in), "
        "or the library's uniform draw on +-1/sqrt(d_in), both pass). Scale the entries so that x @ A keeps "
        "about the size of x")
    again = init_adapter(48, 29, 4, seed=1)
    assert np.array_equal(ad["A"], again["A"]), "the same seed must give the same A"
    assert not np.array_equal(ad["A"], init_adapter(48, 29, 4, seed=2)["A"]), (
        "a different seed must give a different A")
    assert ad["A"].dtype == np.float64 and ad["B"].dtype == np.float64, "use float64"
    ad["B"][0, 0] = 1.0
    assert not np.any(init_adapter(48, 29, 4, seed=1)["B"]), (
        "B must be a fresh array on every call: writing into one adapter's B changed the next adapter's")
    print("exercise 1 looks right: B is zero, A is random with std "
          f"{big.std() * np.sqrt(400):.2f} / sqrt(d_in)")

In [ ]:
if __name__ == "__main__":
    _try("exercise 1", _check_init_adapter)

## 5. Exercises 2 and 3 · the forward pass and the hand-derived backward pass

With a frozen `W`, an adapter `{"A", "B"}` and a scale `s`, the adapted layer is

```text
y = x @ W  +  s * (x @ A) @ B
```

Compute the adapter path as `(x @ A) @ B`, never as `x @ (A @ B)`: the first never builds a
`d_in x d_out` matrix, which is the point of a low-rank update, and the checks measure it (a call
that allocates an array as large as `W` fails, for the forward pass and for the backward pass
alike). The forward pass returns the
output **and a cache** holding what the backward pass needs, with these exact keys: `x`, `W`, `A`,
`B`, `scale`, and `xa` (the product `x @ A`). The cache holds the arrays you were given, not copies
of them: a copy of `W` is the very matrix the adapter exists to avoid duplicating.

**Exercise 2** writes `lora_forward`. **Exercise 3** writes `lora_backward`, the gradient of a
loss with respect to `A`, `B` and the input `x`, given `dy`, the gradient with respect to `y`.
**`W` gets no gradient**: it is frozen, and computing one would defeat the purpose. The input
`x` may have any number of leading axes: a `(batch, time, d_in)` array, a `(batch, d_in)` array
or a single vector. Derive the gradients on paper from the chain rule first (the hints say
where each one comes from), then write them.

<details><summary>💡 Hint 1 — what to think about (exercise 2)</summary>

Treat `x @ A` as an intermediate result, call it `xa`. What is its shape? What does `@ B` turn it
into? Where does the scale go: on `A`, on `B`, or on the finished adapter output? Only one of
those keeps `A` and `B` themselves free of it.

</details>
<details><summary>💡 Hint 2 — the approach in words (exercise 2)</summary>

What does the backward pass need so that it never recomputes anything, and which of those values
were handed to you already? Store references to those, not copies. Then try your function on a
single vector, a matrix and a `(batch, time, d_in)` array: `@` treats the leading axes as batch axes,
so anything that reshapes or transposes by hand is where a shape bug hides. Do not modify anything
you were given.

</details>

In [ ]:
def lora_forward(x: np.ndarray, W: np.ndarray, adapter: dict, scale: float):
    """The adapted layer `y = x @ W + scale * (x @ A) @ B`, and a cache for the backward pass.

    Args:
        x: (..., d_in), any number of leading axes.
        W: (d_in, d_out), frozen.
        adapter: {"A": (d_in, r), "B": (r, d_out)}.
        scale: a float multiplying the adapter path.

    Example:
        >>> x = np.ones((2, 3)); W = np.zeros((3, 4)); ad = {"A": np.ones((3, 1)), "B": np.ones((1, 4))}
        >>> y, cache = lora_forward(x, W, ad, 0.5)
        >>> y.shape, float(y[0, 0])        # 0.5 * (1+1+1) * 1
        ((2, 4), 1.5)

    Returns:
        (y, cache): y has shape (..., d_out); cache is a dict with the keys "x", "W", "A", "B",
        "scale" and "xa" (the array x @ A).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement lora_forward")


def _check_lora_forward() -> None:
    rng = np.random.default_rng(0)
    x, W = rng.normal(size=(3, 5, 6)), rng.normal(size=(6, 4))
    ad = {"A": rng.normal(size=(6, 2)), "B": rng.normal(size=(2, 4))}
    y, cache = lora_forward(x, W, ad, 0.5)
    want = x @ W + 0.5 * (x @ (ad["A"] @ ad["B"]))
    assert y.shape == (3, 5, 4), f"y has shape {y.shape}, expected (3, 5, 4): (..., d_out)"
    assert np.allclose(y, want, rtol=0, atol=1e-12), (
        f"y differs from x @ W + scale * x @ A @ B by up to {np.abs(y - want).max():.2e}: "
        "check the scale and the order of the two products")
    zero = lora_forward(x, W, {"A": ad["A"], "B": np.zeros((2, 4))}, 0.5)[0]
    assert np.allclose(zero, x @ W, rtol=0, atol=1e-12), (
        f"with B = 0 the output must equal x @ W (it differs by up to {np.abs(zero - x @ W).max():.2e}): "
        "the adapter must add nothing at the start")
    nothing = lora_forward(x, np.zeros_like(W), {"A": ad["A"], "B": np.zeros((2, 4))}, 0.5)[0]
    assert not np.any(nothing), (
        "with B = 0 and W = 0 the output must be exactly zero: the adapter path must add exactly nothing "
        "(no constant, no bias) when B is zero")
    d = 1500                                  # a matrix of 18 MB; the adapter and the batch are tiny
    big_x, big_w = rng.normal(size=(2, d)), rng.normal(size=(d, d))
    big_ad = {"A": rng.normal(size=(d, 1)), "B": rng.normal(size=(1, d))}
    peak = _peak_bytes(lambda: lora_forward(big_x, big_w, big_ad, 0.5))
    assert peak < big_w.nbytes // 8, (
        f"lora_forward allocated {peak / 1e6:.1f} MB at one moment on a {d} x {d} matrix ({big_w.nbytes / 1e6:.0f} MB): "
        "it built an array about as large as W. Compute the adapter path as (x @ A) @ B and add it to "
        "x @ W; never form A @ B or W + scale * A @ B, which is the d_in x d_out matrix a low-rank update avoids, "
        "and never copy W (.copy(), .astype(...), np.array(W)): the cache holds the array you were given")
    assert isinstance(cache, dict) and {"x", "W", "A", "B", "scale", "xa"} <= set(cache), (
        "the cache must be a dict with the keys x, W, A, B, scale and xa")
    assert np.allclose(cache["xa"], x @ ad["A"], rtol=0, atol=1e-12), (
        "cache['xa'] must be x @ A, the product the backward pass reuses")
    assert lora_forward(rng.normal(size=6), W, ad, 1.0)[0].shape == (4,), (
        "a single vector x of shape (d_in,) must give shape (d_out,)")
    before = (x.copy(), W.copy(), ad["A"].copy(), ad["B"].copy())
    lora_forward(x, W, ad, 2.0)
    assert all(np.array_equal(a, b) for a, b in zip(before, (x, W, ad["A"], ad["B"]))), (
        "lora_forward modified one of its inputs")
    print("exercise 2 looks right: matches x @ W + scale * x @ A @ B, adds exactly nothing when B = 0, and "
          "builds no matrix as large as W")

In [ ]:
if __name__ == "__main__":
    _try("exercise 2", _check_lora_forward)

### Exercise 3 · `lora_backward(dy, cache)`

Given `dy`, the gradient of the loss with respect to the layer's output `y`, return
`(dx, {"A": dA, "B": dB})`. The three results are the three places the loss can be reached from `y`:
through `x @ W` (to `x` only, since `W` is frozen), and through the adapter path, which has two
factors. The leading axes of `x` and `dy` are batch axes that every gradient of a *weight* sums
over. Run the cell: the check compares your gradients with central finite differences on a
random layer, for a 3-d, a 2-d and a 1-d input.

<details><summary>💡 Hint 1 — what to think about (exercise 3)</summary>

Name the intermediate `xa = x @ A` and write `y = x @ W + s * xa @ B`. The gradient of a product
`p = u @ v` with respect to `u` is `dp @ v.T`, and with respect to `v` is `u.T @ dp`. Apply that
once for `B` (what is `u`, what is `v`?), then once more, through `xa`, for `A` and for `x`.
`x` receives gradient along two routes: sum them.

</details>
<details><summary>💡 Hint 2 — the approach in words (exercise 3)</summary>

Start with `dB`, the simplest: `B` reaches the output only through `s * xa @ B`, so apply the rule
of hint 1 for the right-hand factor of a product, with `xa` as the left factor and the gradient
that arrives at the adapter path as `dp`. Flatten the leading axes of `xa` and of that gradient
into one first, so the transpose is of a 2-d array, and remember that a weight's gradient sums
over every sample. Then use the same rule once more, through `xa = x @ A`: the gradient that
arrives at `xa` is what the first step passed on, and it feeds both `dA` and the adapter's share
of `dx`. `W` gives `dx` a second route; it gives nobody a gradient.

</details>

In [ ]:
def lora_backward(dy: np.ndarray, cache: dict):
    """Gradients of a loss through `lora_forward`, given dy = d loss / d y.

    Args:
        dy: same shape as the layer output y, (..., d_out).
        cache: what `lora_forward` returned: a dict with "x", "W", "A", "B", "scale", "xa".

    Rules:
        * W is frozen: return no gradient for it and never form a (d_in, d_out) gradient;
        * the gradients of A and B sum over every leading (batch) axis;
        * dx has the shape of x and includes both routes from y back to x.

    Example (all shapes; a zero B has a zero gradient for A):
        >>> rng = np.random.default_rng(0)
        >>> x = rng.normal(size=(2, 3)); W = rng.normal(size=(3, 4))
        >>> ad = {"A": rng.normal(size=(3, 2)), "B": np.zeros((2, 4))}
        >>> y, cache = lora_forward(x, W, ad, 1.0)
        >>> dx, g = lora_backward(np.ones_like(y), cache)
        >>> dx.shape, g["A"].shape, g["B"].shape, float(abs(g["A"]).max())
        ((2, 3), (3, 2), (2, 4), 0.0)

    Returns:
        (dx, grads) with grads = {"A": dA, "B": dB}, shaped like A and B.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement lora_backward")


def _numeric_gradient(f, array: np.ndarray, h: float = 1e-6) -> np.ndarray:
    """Central differences of the scalar function f() with respect to every entry of `array`."""
    out = np.zeros_like(array)
    flat, o = array.reshape(-1), out.reshape(-1)
    for i in range(flat.size):
        old = flat[i]
        flat[i] = old + h
        up = f()
        flat[i] = old - h
        down = f()
        flat[i] = old
        o[i] = (up - down) / (2 * h)
    return out


def _cache_for(x, W, adapter, scale) -> dict:
    """The cache exercise 2 is specified to return, built here so that this check (and the merge check
    below) stands on its own: a wrong `lora_forward` cannot make a right `lora_backward` look wrong."""
    return {"x": x, "W": W, "A": adapter["A"], "B": adapter["B"], "scale": scale, "xa": x @ adapter["A"]}


def _check_lora_backward() -> None:
    rng = np.random.default_rng(1)
    for shape in ((2, 3, 5), (4, 5), (5,)):
        x, W = rng.normal(size=shape), rng.normal(size=(5, 4))
        ad = {"A": rng.normal(size=(5, 3)), "B": rng.normal(size=(3, 4))}
        weights = rng.normal(size=shape[:-1] + (4,))

        def loss() -> float:
            return float(((x @ W + 0.7 * ((x @ ad["A"]) @ ad["B"])) * weights).sum())

        cache = _cache_for(x, W, ad, 0.7)
        kept = weights.copy()
        dx, g = lora_backward(weights, cache)
        assert np.array_equal(weights, kept), "lora_backward modified dy, the gradient it was given"
        assert dx.shape == x.shape, f"dx has shape {dx.shape}, expected the shape of x, {x.shape}"
        assert set(g) == {"A", "B"} and g["A"].shape == ad["A"].shape and g["B"].shape == ad["B"].shape, (
            "return (dx, {'A': dA, 'B': dB}) with dA shaped like A and dB shaped like B")
        for name, got, array in (("B", g["B"], ad["B"]), ("A", g["A"], ad["A"]), ("x", dx, x)):
            want = _numeric_gradient(loss, array)
            err = float(np.abs(got - want).max())
            assert err < 1e-6, (
                f"d{name} differs from finite differences by up to {err:.2e} for x of shape "
                f"{shape}. " + {"B": "dB comes from the adapter path alone: check the factor the forward pass "
                                    "applied to that path, and that you sum over every batch axis.",
                                "A": "dA is reached through B: check what the gradient on its way to xa = x @ A "
                                     "was multiplied by, and which factors you transpose.",
                                "x": "dx has two routes back from y, one through W and one through the adapter "
                                     "path: check that both are there and that the second carries what the "
                                     "forward pass applied to that path."}[name])
    zero = {"A": ad["A"], "B": np.zeros_like(ad["B"])}
    _, g0 = lora_backward(weights, _cache_for(x, W, zero, 0.7))
    assert not np.any(g0["A"]) and np.any(g0["B"]), (
        "with B = 0 the gradient of A is exactly zero and the gradient of B is not: A only matters "
        "through B")
    d = 1500                                  # a matrix of 18 MB; the adapter and the batch are tiny
    big_x, big_w, big_dy = rng.normal(size=(2, d)), rng.normal(size=(d, d)), rng.normal(size=(2, d))
    big = _cache_for(big_x, big_w, {"A": rng.normal(size=(d, 1)), "B": rng.normal(size=(1, d))}, 0.5)
    peak = _peak_bytes(lambda: lora_backward(big_dy, big))
    assert peak < big_w.nbytes // 8, (
        f"lora_backward allocated {peak / 1e6:.1f} MB at one moment on a {d} x {d} matrix ({big_w.nbytes / 1e6:.0f} MB): "
        "it built an array about as large as W. W is frozen: return no gradient for it, never form a "
        "d_in x d_out gradient to get dA and dB from it (both come from the thin factors), and never copy W "
        "(.copy(), .astype(...), np.array(W))")
    print("exercise 3 looks right: dA, dB and dx match central finite differences for 3-d, 2-d and 1-d "
          "inputs, and with B = 0 only B receives a gradient")

In [ ]:
if __name__ == "__main__":
    _try("exercise 3", _check_lora_backward)

### What the zero initialisation buys, measured

Three ways to initialise the same adapters on all five matrices, with the base model and a
fixed probe batch of domain B held the same: your rule (`B` zero, `A` random), both zero, and `B`
random with `A` random. The cell asks, for each: how far does the adapted model's output sit from
the base model's *before any training*, and which of `A` and `B` receive a gradient from one
forward and backward pass? Run it, then read the verdict column: it is computed from the numbers
to its left.

In [ ]:
PROBE_SEED = 99            # a fixed batch of 64 windows of held-out domain B for the demos
DEMO_RANK = 8              # the rank of the adapters in the initialisation demo


def _adapters_for(base: dict, rank: int, seed: int, how: str) -> dict:
    """Adapters on all five matrices, started the way `how` says, built from YOUR `init_adapter`."""
    out = {}
    for i, name in enumerate(LAYERS):
        d_in, d_out = base[name].shape
        ad = init_adapter(d_in, d_out, rank, seed=seed + i)
        rng = np.random.default_rng(seed + 50 + i)
        if how == "both zero":
            ad["A"] = np.zeros_like(ad["A"])
        elif how == "B random":
            ad["B"] = rng.normal(0.0, 0.1, ad["B"].shape)
        out[name] = ad
    return out


def _demo_init() -> None:
    probe = windows(B_VAL, np.random.default_rng(PROBE_SEED), 64)
    scale = ALPHA / DEMO_RANK           # the alpha / r scale (exercise 5 writes the general rule)
    base_logits = forward(BASE, {}, 1.0, probe[0])[0]
    base_loss = mean_nll(base_logits, probe[1])
    print(f"the base model on the probe batch: {base_loss:.3f} nats; adapters of rank {DEMO_RANK} on "
          f"{len(LAYERS)} matrices, scale alpha / r = {scale:g}")
    print(f"{'initialisation':<20}{'start loss':>11}{'max |logit change|':>20}{'max |dA|':>11}{'max |dB|':>11}   verdict")
    for how in ("B zero, A random", "both zero", "B random"):
        adapters = _adapters_for(BASE, DEMO_RANK, 7, how)
        logits, cache = forward(BASE, adapters, scale, probe[0])
        _, grads = backward(BASE, adapters, scale, cache, logits, probe[1], False)
        d_a = max(float(np.abs(g["A"]).max()) for g in grads.values())
        d_b = max(float(np.abs(g["B"]).max()) for g in grads.values())
        change = float(np.abs(logits - base_logits).max())
        loss = mean_nll(logits, probe[1])
        if d_a == 0.0 and d_b == 0.0:
            verdict = "dead: no gradient anywhere, it can never leave its start"
        elif change <= 1e-9:
            verdict = "starts as the base model (to rounding), and learns (B first, then A)"
        else:
            verdict = "starts as a different model from the one you are adapting"
        shown = f"{change:.3g}" if change > 1e-9 else "< 1e-9"
        print(f"{how:<20}{loss:>11.3f}{shown:>20}{d_a:>11.1e}{d_b:>11.1e}   {verdict}")

In [ ]:
if __name__ == "__main__":
    _try("what the zero initialisation buys", _demo_init, needs=("exercise 1", "exercise 2", "exercise 3"))

## 6. Exercise 4 · `merge_adapter(W, adapter, scale, unmerge=False)`

At serving time an adapter does not have to stay separate. Because `y = x @ W + s * x @ A @ B` is
`x @ (W + s * A @ B)`, the two can be **merged** into one matrix of the base model's own shape, and the
merged model runs with no extra matrix products and no adapter in sight. The merge can also be undone:
with `unmerge=True` the function subtracts the adapter instead of adding it. This exercise writes both
directions. It returns a *new* matrix: the input `W` must stay untouched, because the point of an
adapter is that one frozen base can serve many of them.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the shape of `A @ B`, and does it match `W`? Which of your inputs must come out of the call
exactly as it went in? Think about what happens to the *next* adapter you want to try if the
base was changed in place.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Form the product of the two thin matrices, multiply it by the scale, and add it to `W`, building
a new array rather than writing into `W`. Unmerging is the same product with the opposite sign.

</details>

In [ ]:
def merge_adapter(W: np.ndarray, adapter: dict, scale: float, unmerge: bool = False) -> np.ndarray:
    """The matrix `W + scale * A @ B` (or `W - scale * A @ B` when `unmerge` is True), as a new
    array: W itself is not modified.

    Example:
        >>> W = np.zeros((2, 2)); ad = {"A": np.ones((2, 1)), "B": np.ones((1, 2))}
        >>> merge_adapter(W, ad, 0.5).tolist()
        [[0.5, 0.5], [0.5, 0.5]]
        >>> merge_adapter(W, ad, 0.5, unmerge=True).tolist()
        [[-0.5, -0.5], [-0.5, -0.5]]
        >>> W.tolist()                       # untouched
        [[0.0, 0.0], [0.0, 0.0]]

    Returns:
        A new (d_in, d_out) float64 array.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement merge_adapter")


def _check_merge_adapter() -> None:
    rng = np.random.default_rng(2)
    x, W = rng.normal(size=(3, 5, 6)), rng.normal(size=(6, 4))
    ad = {"A": rng.normal(size=(6, 2)), "B": rng.normal(size=(2, 4))}
    keep = W.copy()
    merged = merge_adapter(W, ad, 0.5)
    assert merged.shape == W.shape, f"the merged matrix has shape {merged.shape}, expected {W.shape}"
    assert np.array_equal(W, keep), (
        "merge_adapter wrote into W. Build a new array: the base must stay frozen so another adapter "
        "can be merged into it later")
    assert merged is not W and not np.shares_memory(merged, W), "return a new array, not W itself"
    y = x @ W + 0.5 * ((x @ ad["A"]) @ ad["B"])
    assert np.allclose(x @ merged, y, rtol=0, atol=1e-12), (
        f"x @ merged differs from the adapter's output by up to {np.abs(x @ merged - y).max():.2e}: "
        "the adapter path is scale * A @ B, added to W")
    unchanged = merge_adapter(W, {"A": ad["A"], "B": np.zeros((2, 4))}, 0.5)
    assert np.array_equal(unchanged, W), "with B = 0 the merged matrix must equal W exactly"
    scale_zero = merge_adapter(W, ad, 0.0)
    assert np.array_equal(scale_zero, W), "with a scale of 0 the merged matrix must equal W exactly"
    for nothing in (unchanged, scale_zero):
        assert nothing is not W and not np.shares_memory(nothing, W), (
            "even when the adapter adds nothing, return a NEW array: a later write into the merged matrix would "
            "otherwise change the base")
    assert not np.allclose(merge_adapter(W, ad, 0.5), merge_adapter(W, ad, 1.0)), (
        "the scale must change the merged matrix")
    back = merge_adapter(merged, ad, 0.5, unmerge=True)
    assert np.allclose(back, W, rtol=0, atol=1e-12), (
        f"merging and then unmerging gave back a matrix that differs from W by up to "
        f"{np.abs(back - W).max():.2e}: unmerge=True subtracts scale * A @ B")
    assert np.allclose(merge_adapter(W, ad, 0.5, unmerge=True), W - 0.5 * (ad["A"] @ ad["B"]), rtol=0, atol=1e-12), (
        "unmerge=True on W itself must give W - scale * A @ B")
    print("exercise 4 looks right: x @ merged equals the adapter's output, unmerging gets W back to "
          "rounding, and W was left untouched")

In [ ]:
if __name__ == "__main__":
    _try("exercise 4", _check_merge_adapter)

## 7. Exercise 5 · `lora_scale`, and what the scale does to the first steps

The adapter's output is multiplied by a scale `s`, and the scale is a function of the rank. As of
14 October 2026 two rules are in use (sources and quotes in `claims.yaml`):

- **`"lora"`**: `s = alpha / r`, the original paper's rule and PEFT's default. With it,
  doubling the rank halves the scale. PEFT's `LoraConfig` defaults to rank 8 and `lora_alpha` 8,
  which is a scale of exactly 1 at its default rank.
- **`"rslora"`**: `s = alpha / sqrt(r)`, "rank-stabilised" LoRA, which PEFT offers as `use_rslora`.

This exercise writes the function. The cell after it then *measures* what each rule does: the same
model, the same data and the same learning rate, at several ranks, for the first 25 steps, at the two
smallest learning rates of the grid. A third convention, `alpha = r` (a scale of exactly 1), is included
as the case where no scaling is applied at all. Two sources (in `claims.yaml`) make claims about these
rules, and the table puts both to this model:

- **"LoRA Without Regret"** (Thinking Machines Lab, 29 September 2025) reports that with the `alpha / r`
  scale the learning curve is the same at the beginning of training whatever the rank, so that in a
  short run the best learning rate is roughly independent of the rank. It adds that the optimal rate
  "has some rank-dependence in the longer-training regime". The table below looks at the first steps
  only, so look at the loss after 1 and 5 steps across ranks; the sweep of section 10 trains the
  full schedule and tunes the rate for each rank.
- **The rank-stabilised LoRA paper** (Kalajdzievski, 2023) argues that dividing by `r` slows learning
  and stunts higher-rank adapters, and that the factor should be `alpha / sqrt(r)`. That is a claim
  about how far higher ranks get, so look at the loss after 25 steps and at the size of the change.

<details><summary>💡 Hint 1 — what to think about</summary>

Two formulas and two kinds of bad input. What should a rank of zero do? What should a rule name
the function has never heard of do? Both are mistakes to refuse loudly, not to guess at. Also: which
numbers in `alpha / r` could be integers in a notebook, and what does `/` do to them?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check the rank first (a `ValueError` below 1), then branch on the rule name, divide, and return
a Python float. An unknown rule raises `ValueError` too.

</details>

In [ ]:
def lora_scale(alpha: float, rank: int, rule: str = "lora") -> float:
    """The multiplier of the adapter path for rank `rank`.

    Args:
        alpha: the numerator, a number of at least 0 (0 is valid and gives a scale of 0.0).
        rank: the adapter rank, an integer of at least 1.
        rule: "lora" for alpha / rank, "rslora" for alpha / sqrt(rank).

    Raise ValueError for a rank below 1 and for any other rule name (names are case-sensitive).

    Example:
        >>> lora_scale(16, 4, "lora"), lora_scale(16, 4, "rslora")
        (4.0, 8.0)

    Returns:
        A Python float.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement lora_scale")


def _check_lora_scale() -> None:
    got = lora_scale(16, 4, "lora")
    assert abs(got - 4.0) < 1e-12, f"lora_scale(16, 4, 'lora') returned {got!r}; the rule is alpha / rank = 4.0"
    assert type(got) is float, (
        f"lora_scale returned a {type(got).__name__}: return a Python float (float(...), math.sqrt), "
        "not an int and not numpy's float64")
    assert lora_scale(16, 3, "lora") != 16 // 3 and abs(lora_scale(16, 3, "lora") - 16 / 3) < 1e-15, (
        "alpha / rank must be true division: 16 / 3 is not 5")
    got = lora_scale(16, 4, "rslora")
    assert abs(got - 8.0) < 1e-12, f"lora_scale(16, 4, 'rslora') returned {got!r}; the rule is alpha / sqrt(rank) = 8.0"
    assert type(got) is float, (
        f"lora_scale returned a {type(got).__name__} for 'rslora': return a Python float (float(...), "
        "math.sqrt), not an int and not numpy's float64 (np.sqrt returns one)")
    assert abs(lora_scale(10, 1, "rslora") - 10.0) < 1e-12 and abs(lora_scale(10, 1, "lora") - 10.0) < 1e-12, (
        "at rank 1 both rules give alpha")
    for rule in ("lora", "rslora"):
        try:
            zero = lora_scale(0, 4, rule)
        except ValueError as exc:
            raise AssertionError(
                f"lora_scale(0, 4, {rule!r}) raised ValueError({exc}): an alpha of 0 is valid and gives a scale "
                "of 0.0. Refuse only a rank below 1 and a rule name that is not 'lora' or 'rslora'") from None
        assert zero == 0.0, (
            f"lora_scale(0, 4, {rule!r}) returned {zero!r}: an alpha of 0 is valid and gives a scale of 0.0; "
            "do not treat a zero alpha as missing")
    for bad_rank in (0, -2):
        try:
            lora_scale(8, bad_rank, "lora")
        except ValueError:
            pass
        else:
            raise AssertionError(f"rank {bad_rank} must raise ValueError (a zero rank would divide by zero)")
    for bad_rule in ("LoRA", "rs", ""):
        try:
            lora_scale(8, 4, bad_rule)
        except ValueError:
            pass
        else:
            raise AssertionError(f"the rule {bad_rule!r} must raise ValueError: only 'lora' and 'rslora' exist")
    print("exercise 5 looks right: alpha / r, alpha / sqrt(r), and a refusal for a bad rank or rule")

In [ ]:
if __name__ == "__main__":
    _try("exercise 5", _check_lora_scale)

In [ ]:
# What the three conventions do in the first 25 steps. Each cell of the table is one real fine-tune
# of the shared base: constant learning rate, the same batches, a fixed probe batch for the loss.
EARLY_LRS = LRS[:2]                # the two smallest learning rates of the grid
EARLY_RANKS = (1, 4, 16, 32)
CONVENTIONS = (("alpha / r", "lora", lambda r: ALPHA),
               ("alpha / sqrt(r)", "rslora", lambda r: ALPHA),
               ("alpha = r (scale 1)", "lora", lambda r: float(r)))


def update_norm(run: Run) -> float:
    """Frobenius norm of the total weight change `scale * A @ B`, over all adapted matrices."""
    return float(np.sqrt(sum(float(((run.scale * a["A"] @ a["B"]) ** 2).sum()) for a in run.adapters.values())))


TIE_BAND = 1.25     # a winner must be at least this many times flatter than the runner-up to be called


def _flattest(spreads: list, names: list) -> str:
    """The convention with the smallest spread, or "no clear winner" when the runner-up is within
    TIE_BAND times it: one run per cell, so a smaller gap than that is not a ranking."""
    order = sorted(range(len(spreads)), key=lambda i: spreads[i])
    best, second = spreads[order[0]], spreads[order[1]]
    if second < TIE_BAND * best:
        return f"no clear winner ({names[order[0]]} and {names[order[1]]} are within a factor of {TIE_BAND:g})"
    return names[order[0]]


def _demo_early_steps() -> None:
    probe = windows(B_VAL, np.random.default_rng(PROBE_SEED), 64)
    rows = {}                      # (learning rate, convention) -> [(rank, scale, loss@1, loss@5, loss@25, |delta W|)]
    for lr in EARLY_LRS:
        for name, rule, alpha_of in CONVENTIONS:
            rows[lr, name] = []
            for rank in EARLY_RANKS:
                run = finetune(BASE, B_TRAIN, lr=lr, steps=25, seed=2, rank=rank, alpha=alpha_of(rank),
                               rule=rule, decay=False, warmup=0, watch=(1, 5, 25), probe=probe)
                rows[lr, name].append((rank, run.scale, run.watch[1], run.watch[5], run.watch[25],
                                       update_norm(run)))
    lr = EARLY_LRS[0]
    print(f"learning rate {lr:g}, constant, rank in the first column; probe loss in nats "
          f"(the base model scores {mean_nll(forward(BASE, {}, 1.0, probe[0])[0], probe[1]):.3f})")
    for name, _, _ in CONVENTIONS:
        print(f"\n  {name}")
        print(f"  {'rank':>5} {'scale':>7} {'loss@1':>8} {'loss@5':>8} {'loss@25':>8} {'|delta W|':>10}")
        for rank, scale, l1, l5, l25, norm in rows[lr, name]:
            print(f"  {rank:>5} {scale:>7.3f} {l1:>8.3f} {l5:>8.3f} {l25:>8.3f} {norm:>10.2f}")
    print("\nhow much the first step, and the weight change after 25 steps, depend on the rank:")
    for lr in EARLY_LRS:
        spread = {}
        for name, _, _ in CONVENTIONS:
            first = [r[2] for r in rows[lr, name]]
            norms = [r[5] for r in rows[lr, name]]
            spread[name] = (max(first) - min(first), max(norms) / min(norms))
            print(f"  lr {lr:<5g} {name:<22} loss after 1 step varies by {spread[name][0]:.3f} nats across ranks; "
                  f"|delta W| after 25 steps varies by a factor of {spread[name][1]:.2f}")
        print(f"  lr {lr:<5g} flattest first step: {_flattest([spread[n][0] for n in spread], list(spread))}; "
              f"flattest weight change after 25 steps: "
              f"{_flattest([spread[n][1] - 1 for n in spread], list(spread))}")

In [ ]:
if __name__ == "__main__":
    _try("the scaling rules in the first steps", _demo_early_steps, needs=_FOR_TRAINING)

## 8. Exercises 6 and 7 · what an adapter costs, and when it stops being cheap

LoRA's promise is memory. A rank-`r` adapter on a `d_in x d_out` matrix trains `r * (d_in + d_out)`
scalars instead of `d_in * d_out`, and every trainable scalar costs **four** floats of training
state: its value, its gradient, and Adam's two moments (`STATE_COPIES`). The frozen matrix costs
nothing beyond its own storage, which a served model pays anyway.

**Exercise 6** writes the bookkeeping: `adapter_footprint(shapes, rank, itemsize=8)` returns
the trainable count and the state bytes of adapters of one rank on a list of matrices.
**Exercise 7** asks the sharper question: for which ranks is an adapter *smaller* than the matrix
it adapts? `break_even_rank(d_in, d_out)` returns the largest rank whose adapter has strictly fewer
parameters than the matrix. The check runs on hand-computed cases. The cell after that compares your
formula with the bytes of real optimiser state, counted from the arrays of an actual run.

<details><summary>💡 Hint 1 — what to think about (exercise 6)</summary>

Count the scalars in `A` and in `B` for one matrix, add over the matrices, then multiply by the
number of arrays Adam keeps per scalar and by the bytes of one scalar. Keep the two results
separate, in the order the docstring gives.

</details>
<details><summary>💡 Hint 2 — the approach in words (exercise 6)</summary>

Check the docstring example by hand before you run the cell: first how many scalars `A` and `B`
hold for one matrix, then the sum over the list. Of the two numbers you return, which depends on
the size of one float, and which does not? What should an empty list, or a rank of zero, give?

</details>

In [ ]:
def adapter_footprint(shapes: list, rank: int, itemsize: int = 8) -> tuple:
    """Trainable scalars and training-state bytes of rank-`rank` adapters on matrices of `shapes`.

    Args:
        shapes: a list of (d_in, d_out) pairs, one per adapted matrix (may be empty).
        rank: the adapter rank, 0 or more; a negative rank raises ValueError.
        itemsize: bytes per scalar (8 for float64).

    Rules: each adapter holds `rank * (d_in + d_out)` trainable scalars. Training state is
    STATE_COPIES arrays per trainable scalar (the value, its gradient, Adam's first and second
    moment), so state_bytes = STATE_COPIES * itemsize * n_params. The frozen matrices add nothing.

    Example:
        >>> adapter_footprint([(48, 48), (48, 29)], 8)
        (1384, 44288)

    Returns:
        (n_params, state_bytes), two Python ints.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement adapter_footprint")


def _check_adapter_footprint() -> None:
    got = adapter_footprint([(48, 48), (48, 29)], 8)
    assert tuple(got) == (1384, 44288), (
        f"adapter_footprint([(48, 48), (48, 29)], 8) returned {got}, expected (1384, 44288): "
        "8 * (48 + 48) + 8 * (48 + 29) trainable scalars, then 4 arrays of 8-byte floats for each")
    assert all(type(v) is int for v in got), f"return two Python ints, got {[type(v).__name__ for v in got]}"
    n, b = adapter_footprint([(10, 3)], 2, itemsize=4)
    assert (n, b) == (26, 416), (
        f"one 10 x 3 matrix at rank 2 in float32 gives {(n, b)}, expected (26, 416): the byte count "
        "is STATE_COPIES * itemsize * n_params")
    assert tuple(adapter_footprint([], 5)) == (0, 0), "no adapted matrices: no parameters, no state"
    assert tuple(adapter_footprint([(7, 9)], 0)) == (0, 0), "rank 0 is no adapter at all"
    try:
        adapter_footprint([(7, 9)], -1)
    except ValueError:
        pass
    else:
        raise AssertionError("a negative rank must raise ValueError")
    print(f"exercise 6 looks right: {got[0]} adapter scalars on the two example matrices at rank 8, "
          f"and {got[1]} bytes of training state")

In [ ]:
if __name__ == "__main__":
    _try("exercise 6", _check_adapter_footprint)

### Exercise 7 · `break_even_rank(d_in, d_out)`

An adapter of rank `r` has `r * (d_in + d_out)` scalars; the matrix has `d_in * d_out`. For small `r` the adapter is
far smaller, and as `r` grows the two meet and then cross. Return the **largest** `r` such that
`r * (d_in + d_out) < d_in * d_out`, strictly. Use integer arithmetic only, in a closed form with
no loop over ranks and no array of ranks (some inputs are large enough that a loop would never finish
and an array would not fit in memory, which the check tests before it tries them): the answer is an
integer, and on a matrix where the two counts can be exactly equal, a float rounding error or a `<=`
in the wrong place gives a rank that is one too high. Python integers never overflow; numpy's do.

<details><summary>💡 Hint 1 — what to think about</summary>

Rearrange the inequality to get `r` on one side. The right-hand side is a fraction. For a square
matrix of width 48, what rank makes the two counts exactly equal, and is that rank allowed?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

`r` must be an integer strictly below the fraction `d_in * d_out / (d_in + d_out)`. Write down the answer
for the two cases, a fraction that is a whole number and one that is not, then ask whether one
integer-only expression can give both. Do not go through a float, and do not build an array of ranks.

</details>

In [ ]:
def break_even_rank(d_in: int, d_out: int) -> int:
    """The largest rank r whose adapter has strictly fewer parameters than the matrix:
    r * (d_in + d_out) < d_in * d_out. Returns 0 when not even rank 1 qualifies (a 2 x 2 matrix).

    Raise ValueError if d_in or d_out is below 1. Python integer arithmetic only, with no loop over
    ranks and no array of ranks: some inputs are large enough that a loop would never finish and an
    array would not fit in memory (numpy integers also overflow at 2**63).

    Example:
        >>> break_even_rank(48, 29), break_even_rank(48, 48)
        (18, 23)

    Returns:
        A Python int, 0 or more.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement break_even_rank")


_BREAK_EVEN_WHY = ("a loop over every rank needs about 10**8 steps on this matrix; the answer is a closed form "
                   "in Python integers")


def _check_break_even_rank() -> None:
    for (d_in, d_out), want in (((48, 29), 18), ((48, 48), 23), ((4, 4), 1), ((2, 2), 0), ((1, 9), 0),
                                ((6, 3), 1), ((100, 1), 0)):
        got = break_even_rank(d_in, d_out)
        assert got == want and type(got) is int, (
            f"break_even_rank({d_in}, {d_out}) returned {got!r}, expected {want}: it is the largest "
            f"integer r with r * {d_in + d_out} strictly below {d_in * d_out}. "
            + ("When the product divides exactly, equality is not 'fewer'." if (d_in * d_out) % (d_in + d_out) == 0
               else "Round down, not to the nearest integer."))
    peak = _peak_bytes(lambda: break_even_rank(10 ** 6, 10 ** 6))
    assert peak < 100_000, (
        f"break_even_rank(10**6, 10**6) allocated {peak / 1e6:.1f} MB at one moment: building an array (or a list) "
        "of candidate ranks needs memory in proportion to the matrix size, and on the matrices below it would "
        "need gigabytes. The answer is a closed form in Python integers")
    with _time_limit(1, "break_even_rank(100020000, 500100000000)", _BREAK_EVEN_WHY):
        big = break_even_rank(100020000, 500100000000)
    assert big == 10 ** 8 - 1, (
        f"break_even_rank(100020000, 500100000000) returned {big}: the two counts tie exactly at rank 10**8, "
        "which is not 'fewer'. A float division cannot see the tie, and numpy integers overflow on a product "
        "this large (it is above 2**63); use Python integers only")
    with _time_limit(1, "break_even_rank(100903169, 11172124929)", _BREAK_EVEN_WHY):
        big = break_even_rank(100903169, 11172124929)
    assert big == 10 ** 8, (
        f"break_even_rank(100903169, 11172124929) returned {big}: the product exceeds 10**8 times the sum by "
        "exactly 1, so rank 10**8 still has fewer parameters than the matrix; a float division loses that 1")
    for bad in ((0, 4), (4, 0), (-1, 5)):
        try:
            break_even_rank(*bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"break_even_rank{bad} must raise ValueError")
    print(f"exercise 7 looks right: {break_even_rank(48, 48)} for a 48 x 48 matrix (the next rank would "
          f"tie, not save), {break_even_rank(48, 29)} for 48 x 29")

In [ ]:
if __name__ == "__main__":
    _try("exercise 7", _check_break_even_rank)

In [ ]:
def _demo_footprint() -> None:
    shapes = [BASE[n].shape for n in LAYERS]
    full = finetune(BASE, B_TRAIN, lr=1e-2, steps=3, seed=2)
    print(f"every trainable scalar costs {STATE_COPIES} floats of training state; the numbers below "
          "are summed from the arrays of real runs")
    print(f"{'':<14}{'trainable':>10}{'state bytes':>13}{'share of full':>15}{'formula agrees':>16}")
    print(f"{'full fine-tune':<14}{full.trainable:>10}{full.state_bytes:>13}{'100.0%':>15}{'':>16}")
    for rank in RANKS:
        run = finetune(BASE, B_TRAIN, lr=1e-2, steps=3, seed=2, rank=rank)
        n, b = adapter_footprint(shapes, rank)
        agree = (n, b) == (run.trainable, run.state_bytes)
        print(f"{'rank ' + str(rank):<14}{run.trainable:>10}{run.state_bytes:>13}"
              f"{100 * run.state_bytes / full.state_bytes:>14.1f}%{str(agree):>16}")
    wide = ", ".join(f"d = {d}: {100 * adapter_footprint([(d, d)], 8)[0] / (d * d):.2f}%" for d in (48, 768, 4096))
    print(f"\nthe same rank-8 adapter as a share of the one square matrix it adapts: {wide}")
    print("break-even ranks (largest rank with fewer parameters than the matrix): "
          + ", ".join(f"{d_in}x{d_out}: {break_even_rank(d_in, d_out)}"
                      for d_in, d_out in ((48, 48), (48, 29), (768, 768), (4096, 4096), (4096, 11008))))
    over = [r for r in RANKS if r > min(break_even_rank(*BASE[n].shape) for n in LAYERS)]
    print(f"ranks in this lesson's sweep that cost more than the matrix on at least one of its five "
          f"matrices: {over if over else 'none'}")

In [ ]:
if __name__ == "__main__":
    _try("measured footprint", _demo_footprint,
         needs=_FOR_TRAINING + ("exercise 6", "exercise 7"))

## 9. Exercise 8 · `find_knee(ranks, losses, margin_milli)`

A rank sweep gives one loss per rank, and "the rank at which the adapter stops helping" needs an
operational definition before it can be computed. This lesson's: **the knee is the smallest rank
whose loss is within a margin of the best loss in the sweep.**

Floating-point edges would make that comparison unreliable (`0.20 - 0.15 > 0.05` is `True` in
binary), so the comparison is done in **whole milli-nats**: turn every loss into the integer
`round(loss * 1000)` (Python's own `round`, which sends an exact half to the even integer), and
only then compare: a rank qualifies when its integer is at most the smallest integer plus
`margin_milli`. Two losses that straddle a rounding boundary can therefore sit one milli-nat
apart even though they differ by less than a thousandth of a nat. Ranks arrive in any order,
and a run that diverged (a loss of `nan` or `inf`) can be neither the best nor the knee.

<details><summary>💡 Hint 1 — what to think about</summary>

Which losses count at all? Find the best one first, then the set of ranks that are close enough,
then pick from that set. Two ranks can qualify at once: which of them is the knee? And is the
list you were given guaranteed to be in rank order?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Pair each rank with its loss, drop the pairs whose loss is not finite, convert the rest to whole
milli-nats, take the minimum, keep the ranks whose value is at most that minimum plus the margin,
and return the smallest rank among them. Refuse (`ValueError`) mismatched or empty lists, a negative
margin, and a sweep with no finite loss at all.

</details>

In [ ]:
def find_knee(ranks: list, losses: list, margin_milli: int) -> int:
    """The smallest rank whose loss is within `margin_milli` milli-nats of the best loss.

    Args:
        ranks: distinct positive integers, in any order; losses[i] belongs to ranks[i].
        losses: floats in nats; nan and inf mean a diverged run and are ignored.
        margin_milli: a non-negative integer, in thousandths of a nat.

    Rules: milli(l) = round(l * 1000), an integer, with Python's built-in round (an exact half goes
    to the even integer: round(1062.5) is 1062). Let best = the smallest milli(l) over the finite
    losses. A rank qualifies when milli(its loss) <= best + margin_milli (at equality it qualifies):
    round each loss FIRST, then compare integers. Return the smallest qualifying RANK (not the first
    in the list).

    Raise ValueError if the lists are empty or of different length, if margin_milli is negative,
    or if no loss is finite.

    Example:
        >>> find_knee([1, 2, 4, 8], [1.500, 1.100, 1.040, 1.000], 50)
        4
        >>> find_knee([8, 1, 4, 2], [1.000, 1.500, 1.040, 1.100], 50)    # the same, shuffled
        4

    Returns:
        A Python int, one of `ranks`.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement find_knee")


def _check_find_knee() -> None:
    got = find_knee([1, 2, 4, 8], [1.500, 1.100, 1.040, 1.000], 50)
    assert got == 4, (
        f"find_knee returned {got!r}; the best loss is 1.000, 50 milli-nats of margin allows 1.050, "
        "and rank 4 (1.040) is the smallest rank that qualifies")
    assert type(got) is int, (
        f"find_knee returned a {type(got).__name__}: return a Python int (int(...) of the rank), "
        "not a numpy integer")
    got = find_knee([8, 1, 4, 2], [1.000, 1.500, 1.040, 1.100], 50)
    assert got == 4, (
        f"on the same curve in a different order find_knee returned {got!r}, expected 4: return the "
        "smallest qualifying RANK, not the first qualifying entry of the list")
    assert find_knee([1, 2, 4], [1.100, 1.050, 1.000], 50) == 2, (
        "a loss exactly one margin above the best (1.050 against 1.000, margin 50) qualifies: "
        "the comparison is 'at most'")
    assert find_knee([1, 2, 4], [1.101, 1.051, 1.000], 50) == 4, (
        "1.051 is 51 milli-nats above the best and must not qualify at margin 50")
    assert find_knee([1, 2, 4, 8], [1.20, 1.00, 1.10, 0.90], 0) == 8, (
        "with margin 0 only the best itself qualifies, even when the curve is not monotone")
    assert find_knee([1, 2, 4], [float("nan"), 2.0, 1.0], 50) == 4, (
        "a diverged run (nan) is ignored: it is neither the best nor a candidate")
    assert find_knee([1, 2, 4], [float("inf"), 1.0, 1.0], 50) == 2, "inf is ignored as well"
    assert find_knee([2, 1], [0.0, 0.2], 10) == 2, (
        "a loss of exactly 0.0 is a real loss, and here the best one: test whether a loss is finite, not "
        "whether it is truthy")
    assert find_knee([1, 2, 4], [1.0505001, 1.0494999, 1.0], 50) == 2, (
        "round(loss * 1000) rounds to the NEAREST milli-nat: 1.0505001 is 1051, which is more than 50 above "
        "1000, and 1.0494999 is 1049, which is not. int() would truncate both to 1050")
    assert find_knee([1, 2], [1.0503, 1.0], 50) == 1 and find_knee([1, 2], [1.0506, 1.0004], 50) == 2, (
        "round each loss to whole milli-nats first, then compare the integers: 1.0503 is 1050 and 1.0 is 1000, "
        "so they differ by exactly 50 and qualify (a float difference of 0.0503 does not); 1.0506 is 1051 and "
        "1.0004 is 1000, so they differ by 51 and do not (although the float difference, 50.2 milli-nats, "
        "rounds to 50)")
    assert find_knee([1, 2], [1.0625, 1.0], 62) == 1 and find_knee([1, 2], [1.1875, 1.0], 187) == 2, (
        "1.0625 * 1000 is exactly 1062.5 and Python's round sends an exact half to the even integer, 1062: "
        "that is 62 above 1000 and qualifies at margin 62; 1.1875 gives 1187.5, which goes up to 1188: 188 "
        "above 1000, so it does not qualify at margin 187")
    assert find_knee([1, 2], [0.999, 0.939], 60) == 1, (
        "0.999 is exactly 60 milli-nats above 0.939 and qualifies at margin 60, although the float sum "
        "0.939 + 0.06 is a hair below 0.999: compare whole milli-nats")
    for args in (([], [], 50), ([1, 2], [1.0], 50), ([1], [1.0], -1), ([1, 2], [float("nan"), float("nan")], 50)):
        try:
            find_knee(*args)
        except ValueError:
            pass
        else:
            raise AssertionError(f"find_knee{args} must raise ValueError")
    print("exercise 8 looks right: smallest rank within the margin of the best, in milli-nats, "
          "order-independent, ignoring diverged runs")

In [ ]:
if __name__ == "__main__":
    _try("exercise 8", _check_find_knee)

## 10. The rank sweep

Now the experiment the lesson is named for. For every rank in `RANKS` and every learning rate in
`LRS`, and for each of the seeds in `SEEDS`, the cell fine-tunes adapters on all five matrices
(scale `alpha / r`, the PEFT default rule) for the same `STEPS` steps, and scores them on the
validation and test sets and on held-out domain A. Then, in the order a careful experimenter would:

1. **choose each rank's learning rate on validation**, averaging the seeds;
2. **find the knee on validation** with your `find_knee`, and read the test loss only to report;
3. **repeat the knee at each fixed learning rate**, to see what a sweep at one untuned learning
   rate would have told you instead.

It also prints the seed-to-seed spread, because a knee smaller than the noise is not a finding.
The sweep is the lesson's longest cell. Everything it prints is computed from the runs.

**What the knee is, and is not.** A rank-32 adapter can represent every update a rank-8 adapter can
(set the extra columns of `A` and rows of `B` to zero), so a loss that is higher at the larger rank is
never a lack of capacity. "LoRA Without Regret" (Thinking Machines Lab, 29 September 2025; quotes in
`claims.yaml`) says LoRA has enough capacity for the dataset sizes typical of post-training and falls
short when the data are far larger; this sweep cannot test that. What it measures is the **knee of this
protocol**: the rank beyond which a bigger adapter stopped lowering the validation loss under this
schedule, this grid and this scale rule. The post also reports that the best learning rate for LoRA was
consistently about ten times the best one for full fine-tuning in its experiments, so the final lines of
the cell print the same ratio for this grid and say whether the grid can show one. The cell after the
sweep repeats it with the other scale rule.

In [ ]:
MARGIN_MILLI = 50          # the knee margin: 0.050 nats on validation
SWEEP = None               # filled in by the sweep cell: {(rank, lr, seed): (run, val, test, held-out A)}
KNEE, KNEE_RUN = None, None  # the knee rank, and one adapter run at that rank, set by the sweep cell
MATCH_RANK = PEFT_DEFAULTS["r"]                        # the rank at which the two scale rules are made equal
ALPHA_RS = ALPHA / float(np.sqrt(MATCH_RANK))          # alpha / sqrt(r) with this alpha equals ALPHA / r at MATCH_RANK


def run_sweep(rule: str = "lora", alpha: float = ALPHA) -> dict:
    """Every (rank, lr, seed) fine-tune of the sweep, scored on validation, test and domain A."""
    runs = {}
    for rank in RANKS:
        for lr in LRS:
            for seed in SEEDS:
                run = finetune(BASE, B_TRAIN, lr=lr, seed=seed, rank=rank, alpha=alpha, rule=rule)
                runs[rank, lr, seed] = (run, score(run, B_VAL), score(run, B_TEST), score(run, A_TEST))
    return runs


def _col(runs: dict, rank, lr, column: int) -> float:
    return float(np.mean([runs[rank, lr, s][column] for s in SEEDS]))


def tuned_lr(runs: dict, rank) -> float:
    """The learning rate with the lowest mean validation loss for this rank."""
    return min(LRS, key=lambda lr: _col(runs, rank, lr, 1))


def _milli(x: float) -> int:
    """A loss, or a difference of losses, in whole milli-nats: Python's round, as find_knee uses."""
    return round(1000 * x)


def _compare(a: int, b: int) -> str:
    """The three outcomes of comparing two whole numbers of milli-nats."""
    return "more than" if a > b else "equal to" if a == b else "less than"


def _past_the_knee(label: str, val: dict, spread: dict, knee: int) -> list:
    """Print each larger rank's mean validation loss minus the knee's, in milli-nats, next to the larger of
    the two ranks' seed ranges. Returns the ranks that sit above the knee's loss by more than that range."""
    above = {r: (_milli(val[r]) - _milli(val[knee]), max(spread[r], spread[knee])) for r in RANKS if r > knee}
    if not above:
        print(f"  {label}: the knee is the largest rank tried")
        return []
    print(f"  {label}, larger ranks minus the knee's loss (rank {knee}), in milli-nats, with the larger seed range of "
          "the pair in brackets: " + ", ".join(f"r={r}: {d:+d} ({band})" for r, (d, band) in above.items()))
    return [r for r, (d, band) in above.items() if d > band]


def _demo_sweep() -> None:
    global SWEEP, KNEE, KNEE_RUN
    SWEEP = runs = run_sweep()
    print(f"{len(RANKS)} ranks x {len(LRS)} learning rates x {len(SEEDS)} seeds = {len(runs)} adapter "
          f"fine-tunes of {STEPS} steps; full fine-tuning used the same grid and seeds")
    print(f"scale alpha / r with alpha = {ALPHA:g}; PEFT's LoraConfig defaults as of 14 October 2026 are "
          f"r = {PEFT_DEFAULTS['r']}, lora_alpha = {PEFT_DEFAULTS['lora_alpha']} (a scale of "
          f"{PEFT_DEFAULTS['lora_alpha'] / PEFT_DEFAULTS['r']:g} at its default rank) and "
          f"lora_dropout = {PEFT_DEFAULTS['lora_dropout']}, which this lesson does not use")
    full_val = {lr: float(np.mean([FULL[lr, s][1] for s in SEEDS])) for lr in LRS}
    full_test = float(np.mean([FULL[FULL_LR, s][2] for s in SEEDS]))
    print(f"\n{'':<10}{'trainable':>10}{'state bytes':>13}{'lr':>8}{'val':>8}{'test':>8}{'val seed range':>16}")
    full_run = FULL[FULL_LR, SEEDS[0]][0]
    print(f"{'full FT':<10}{full_run.trainable:>10}{full_run.state_bytes:>13}{FULL_LR:>8.0e}"
          f"{full_val[FULL_LR]:>8.3f}{full_test:>8.3f}{'':>16}")
    tuned = {}
    for rank in RANKS:
        lr = tuned_lr(runs, rank)
        vals = [runs[rank, lr, s][1] for s in SEEDS]
        run = runs[rank, lr, SEEDS[0]][0]
        tuned[rank] = (lr, float(np.mean(vals)), _col(runs, rank, lr, 2), max(vals) - min(vals), run)
        print(f"{'rank ' + str(rank):<10}{run.trainable:>10}{run.state_bytes:>13}{lr:>8.0e}"
              f"{tuned[rank][1]:>8.3f}{tuned[rank][2]:>8.3f}{tuned[rank][3]:>16.3f}")
    val = {r: tuned[r][1] for r in RANKS}
    spread = {r: _milli(tuned[r][3]) for r in RANKS}
    knee = find_knee(list(RANKS), [val[r] for r in RANKS], MARGIN_MILLI)
    best_test = min(tuned[r][2] for r in RANKS)
    knee_run = tuned[knee][4]
    full_run = FULL[FULL_LR, SEEDS[0]][0]
    KNEE, KNEE_RUN = knee, knee_run
    print(f"\nthe knee, on validation, at {MARGIN_MILLI} milli-nats of margin: rank {knee} "
          f"(learning rate {tuned[knee][0]:.0e})")
    print(f"  test loss at the knee {tuned[knee][2]:.3f} nats; best test loss in the sweep {best_test:.3f}; "
          f"full fine-tuning {full_test:.3f}")
    print(f"  at the knee the adapters train {100 * knee_run.trainable / full_run.trainable:.1f}% of the "
          f"parameters and hold {100 * knee_run.state_bytes / full_run.state_bytes:.1f}% of the training "
          f"state of full fine-tuning")
    print(f"  the two seeds' validation losses at the knee differ by {spread[knee]} milli-nats "
          f"({_compare(spread[knee], MARGIN_MILLI)} the {MARGIN_MILLI} milli-nat margin)")
    if knee == max(RANKS):
        print("  the knee is the largest rank tried: this sweep never saw the curve flatten")
    # What one untuned learning rate would have said.
    print(f"\nthe same knee at each FIXED learning rate (validation loss by rank, mean of {len(SEEDS)} seeds):")
    print(f"{'lr':>8}  " + "".join(f"{'r=' + str(r):>8}" for r in RANKS) + "   knee")
    knees = {}
    for lr in LRS:
        row = [_col(runs, r, lr, 1) for r in RANKS]
        knees[lr] = find_knee(list(RANKS), row, MARGIN_MILLI)
        print(f"{lr:>8.0e}  " + "".join(f"{v:>8.3f}" for v in row) + f"   rank {knees[lr]}")
    print(f"knee ranks at the {len(LRS)} fixed learning rates: {sorted(set(knees.values()))}; "
          f"with each rank's own learning rate: {knee}")
    all_knees = set(knees.values()) | {knee}
    print("so the rank at which the adapters 'stop helping' "
          + ("depends on the learning rate the sweep happened to use" if len(all_knees) > 1
             else "was the same under every learning rate on this grid")
          + f"; across these learning rates it moves by "
          f"{max(RANKS.index(k) for k in all_knees) - min(RANKS.index(k) for k in all_knees)} step(s) of the "
          "rank grid (each step doubles the rank)")
    edge = [r for r in RANKS if tuned[r][0] in (min(LRS), max(LRS))]
    print("ranks whose best learning rate sits on the edge of the grid (the optimum may lie beyond it): "
          + (str(edge) if edge else "none"))
    print("best adapter learning rate over the full fine-tune's, by rank (the cited post reports about 10x at scale): "
          + ", ".join(f"r={r}: {tuned[r][0] / FULL_LR:.1f}x" for r in RANKS))
    if FULL_LR in (min(LRS), max(LRS)) or edge:
        print("at least one of those best rates sits on the edge of the grid, so each ratio is set by the grid, "
              "not measured: this sweep cannot say whether the cited ratio holds for this model")
    # What kind of knee this is.
    larger = [r for r in RANKS if r >= knee]
    on_edge = [r for r in larger if r in edge]
    if on_edge:
        which = ("the knee and of every larger rank" if on_edge == larger
                 else f"rank(s) {on_edge}, among the knee and the larger ranks,")
        print(f"\nso this is not a tuned knee: the best learning rate of {which} sits on the edge of the grid. It is "
              "the knee at the best reproducible rate, and a rate beyond the grid could lower some ranks' losses by "
              "more than others' and move it")
    higher = _past_the_knee("alpha / r", val, spread, knee)
    if higher:
        print(f"  rank(s) {higher} sit above the knee's loss by more than the seeds' range. A rank-{max(RANKS)} adapter "
              f"can represent every update a rank-{knee} adapter can, so that is not a lack of capacity: it is what the "
              f"optimiser did with the extra parameters in {STEPS} steps at this rate (or overfitting). With the scale "
              "alpha / r the step of the update shrinks as the rank grows, which is the claim of the rank-stabilised LoRA "
              "paper in section 7; the next cell tests it")
    elif knee < max(RANKS):
        print("  no larger rank is above the knee's loss by more than the seeds' range: as far as two seeds can tell, "
              "the curve is flat beyond the knee")

In [ ]:
if __name__ == "__main__":
    _try("the rank sweep", _demo_sweep, needs=_FOR_TRAINING + ("exercise 8",))

### Does the knee depend on the scale rule?

Section 7 quoted the rank-stabilised LoRA paper's claim that dividing the adapter by `r` slows learning
and stunts the performance of higher-rank adapters. If that were what holds the larger ranks back here,
the same sweep with `alpha / sqrt(r)` would change the curve beyond the knee. To compare the two rules at
one overall size, this `alpha` is chosen so that the two scales are equal at rank `MATCH_RANK` (PEFT's
default rank): below it the new scale is smaller, above it larger. The cell reruns every fine-tune of
the sweep with it and prints the knee at each rate, the curve beyond the knee, and the difference between
the two rules at each rank next to the seed range. A difference smaller than the seed range of two seeds
is not a finding in either direction.

In [ ]:
SWEEP_RS = None       # the same sweep with the scale alpha / sqrt(r), set by the next cell


def _demo_scale_rule() -> None:
    global SWEEP_RS
    SWEEP_RS = rs = run_sweep("rslora", ALPHA_RS)
    lo = SWEEP
    print(f"the same {len(rs)} fine-tunes with the scale alpha / sqrt(r) and alpha = {ALPHA_RS:.3f}: at rank "
          f"{MATCH_RANK} that scale is {lora_scale(ALPHA_RS, MATCH_RANK, 'rslora'):.3f}, the same as alpha / r with "
          f"alpha = {ALPHA:g} gives there ({lora_scale(ALPHA, MATCH_RANK, 'lora'):.3f}); above rank {MATCH_RANK} it is "
          "larger, below it smaller")
    print(f"validation loss by rank at each FIXED learning rate (mean of {len(SEEDS)} seeds), scale alpha / sqrt(r):")
    print(f"{'lr':>8}  " + "".join(f"{'r=' + str(r):>8}" for r in RANKS) + "   knee")
    knees_rs, knees_lo = {}, {}
    for lr in LRS:
        row = [_col(rs, r, lr, 1) for r in RANKS]
        knees_rs[lr] = find_knee(list(RANKS), row, MARGIN_MILLI)
        knees_lo[lr] = find_knee(list(RANKS), [_col(lo, r, lr, 1) for r in RANKS], MARGIN_MILLI)
        print(f"{lr:>8.0e}  " + "".join(f"{v:>8.3f}" for v in row) + f"   rank {knees_rs[lr]}")
    print(f"\neach rank at its own best learning rate (mean of {len(SEEDS)} seeds; range = the two seeds' range in milli-nats)")
    print(f"{'':>6}  {'alpha / r':^23}  {'alpha / sqrt(r)':^23}  {'alpha / sqrt(r)':>15}")
    print(f"{'rank':>6}  {'lr':>7}{'val':>8}{'range':>8}  {'lr':>7}{'val':>8}{'range':>8}  {'minus alpha / r':>15}")
    val_lo, val_rs, spread_lo, spread_rs, rows = {}, {}, {}, {}, []
    for rank in RANKS:
        out = []
        for runs, val, spread in ((lo, val_lo, spread_lo), (rs, val_rs, spread_rs)):
            lr = tuned_lr(runs, rank)
            vals = [runs[rank, lr, s][1] for s in SEEDS]
            val[rank], spread[rank] = float(np.mean(vals)), _milli(max(vals) - min(vals))
            out.append(lr)
        diff = _milli(val_rs[rank]) - _milli(val_lo[rank])
        rows.append((rank, diff, max(spread_lo[rank], spread_rs[rank])))
        print(f"{rank:>6}  {out[0]:>7.0e}{val_lo[rank]:>8.3f}{spread_lo[rank]:>8}  {out[1]:>7.0e}{val_rs[rank]:>8.3f}"
              f"{spread_rs[rank]:>8}  {diff:>+15d}")
    knee_lo = find_knee(list(RANKS), [val_lo[r] for r in RANKS], MARGIN_MILLI)
    knee_rs = find_knee(list(RANKS), [val_rs[r] for r in RANKS], MARGIN_MILLI)
    print(f"\nknee ranks at the {len(LRS)} fixed learning rates: alpha / r {[knees_lo[lr] for lr in LRS]}, "
          f"alpha / sqrt(r) {[knees_rs[lr] for lr in LRS]}; with each rank's own rate: {knee_lo} and {knee_rs}")
    moved = [f"{lr:.0e}" for lr in LRS if knees_lo[lr] != knees_rs[lr]] + (["each rank's own rate"] if knee_lo != knee_rs else [])
    print("the knee is " + ("the same under both rules at every learning rate and with each rank's own rate"
                            if not moved else f"different under the two rules at: {', '.join(moved)}"))
    _past_the_knee("alpha / sqrt(r)", val_rs, spread_rs, knee_rs)
    lower_rs = [r for r, d, band in rows if -d > band]
    lower_lo = [r for r, d, band in rows if d > band]
    if lower_rs or lower_lo:
        print(f"the rules differ by more than the larger of their seed ranges at rank(s) {sorted(lower_rs + lower_lo)}: "
              f"alpha / sqrt(r) is lower at {lower_rs or 'none'}, alpha / r is lower at {lower_lo or 'none'}")
    else:
        print("at no rank do the two rules differ by more than the larger of their seed ranges: with two seeds this "
              "sweep does not separate them")

In [ ]:
if __name__ == "__main__":
    _try("the rank sweep under alpha / sqrt(r)", _demo_scale_rule,
         needs=_FOR_TRAINING + ("exercise 8", "the rank sweep"))

## 11. Exercise 9 · `effective_rank(delta, energy)`: how many directions did full fine-tuning use?

The LoRA paper hypothesises that the change a fine-tune makes to a weight matrix is concentrated in a few directions. The
singular value decomposition of that change, `delta_W = W_finetuned - W_base`, says how concentrated
it is: the squared singular values are the energy of each direction. The **effective rank at
energy `e`** is the smallest number of directions whose energy adds up to a fraction `e` of the total.

To keep the comparison free of floating-point edges, round every cumulative fraction to **6 decimal
places** before comparing it with `energy`. A zero matrix has no directions: its effective rank is 0.
The cell after the exercise applies your function to the update full fine-tuning made to each of
the five matrices, and, if the rank sweep has run, to the update the adapter at the knee made.
"LoRA Learns Less and Forgets Less" (Biderman et al., 2024; `claims.yaml`) reports that full
fine-tuning of large models learns updates whose rank is 10 to 100 times greater than typical LoRA
ranks; this model is far smaller, and the cell prints what it does.

<details><summary>💡 Hint 1 — what to think about</summary>

Which quantity of the decomposition is "energy": the singular values or their squares? Sorted in which
order? And what should the answer be for a matrix that is all zeros, where dividing by the total
energy would divide by zero?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Take the singular values of `delta` without the vectors, square them, and form the running sum
divided by the total. Round that to 6 places, and find the first position whose value is at least
`energy`; the rank is that position counted from 1. Refuse a `delta` that is not 2-d and an `energy`
outside (0, 1].

</details>

In [ ]:
def effective_rank(delta: np.ndarray, energy: float) -> int:
    """Smallest k such that the k largest squared singular values hold at least `energy` of the total.

    Args:
        delta: a 2-d array.
        energy: a fraction in (0, 1].

    Rules: s = singular values of delta in descending order; the cumulative fraction after k
    directions is sum(s[:k] ** 2) / sum(s ** 2), rounded to 6 decimal places with round(f, 6) or
    np.round(f, 6) (either: they differ only for a fraction within about one unit in the last place
    of a rounding half, and no graded case is that close); the answer is the smallest k whose
    rounded fraction is >= energy (so a fraction that rounds to exactly `energy` counts). A matrix
    whose entries are all zero has effective rank 0 (decide that exactly: a matrix with entries of the
    order of 1e-12 is not zero).

    Raise ValueError if delta is not 2-d or energy is not in (0, 1].

    Example:
        >>> effective_rank(np.diag([3.0, 1.0, 1.0, 1.0]), 0.5)      # 9 of the 12 units of energy
        1
        >>> effective_rank(np.diag([3.0, 1.0, 1.0, 1.0]), 0.9)
        3

    Returns:
        A Python int between 0 and min(delta.shape).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement effective_rank")


def _check_effective_rank() -> None:
    d = np.diag([3.0, 1.0, 1.0, 1.0])
    for energy, want in ((0.5, 1), (0.75, 1), (0.9, 3), (1.0, 4)):
        got = effective_rank(d, energy)
        assert got == want and type(got) is int, (
            f"effective_rank(diag(3, 1, 1, 1), {energy}) returned {got!r}, expected {want}: the energies "
            "are 9, 1, 1, 1 (the SQUARES of the singular values), and the running fractions are 0.75, "
            "0.833, 0.917, 1")
    assert effective_rank(np.diag([3.0, 1.0, 1e-4]), 1.0) == 2, (
        "round each running fraction to 6 decimal places before comparing: after two directions the fraction "
        "is 0.999999999, which rounds to 1.0, so full energy needs only two")
    rng = np.random.default_rng(3)
    u, _ = np.linalg.qr(rng.normal(size=(20, 20)))
    v, _ = np.linalg.qr(rng.normal(size=(15, 15)))
    s = np.zeros(15)
    s[:5] = [5.0, 4.0, 3.0, 2.0, 1.0]
    delta = (u[:, :15] * s) @ v.T                                   # rank 5, singular values known
    assert effective_rank(delta, 0.999) == 5 and effective_rank(delta, 1.0) == 5, (
        "a matrix of rank 5 has no energy beyond its fifth direction, so it needs exactly 5 for "
        "full energy: the rank of a 20 x 15 matrix is not min(shape)")
    assert effective_rank(delta, 0.6) == 2, (
        "the energies are 25, 16, 9, 4, 1: two directions hold 41 of 55 (0.745), one holds only 0.455")
    assert effective_rank(1e-12 * delta, 0.999) == 5 and effective_rank(1e-12 * delta, 0.6) == 2, (
        "a matrix of tiny entries (1e-12 times the one above) has the same directions in the same proportions: "
        "energy is a fraction of the total, and 'all zero' means exactly zero, not close to zero")
    for tail, energy, want, why in (
            (6.3e-3, 1.0, 3, "0.999996 after two directions: that is below 1.0 once rounded to 6 places"),
            (2e-3, 1.0, 2, "0.9999996 after two directions: that rounds to 1.0 at 6 places, not at 7"),
            (2.65e-3, 1.0, 3, "0.9999993 after two directions: that rounds to 0.999999, below 1.0; "
                              "a tolerance of 1e-6 would call it 1.0"),
            (6.3e-3, 0.999996, 2, "0.999996 after two directions: a fraction that rounds to exactly the "
                                  "energy counts, and the energy itself is not rounded")):
        got = effective_rank(np.diag([3.0, 1.0, tail]), energy)
        assert got == want, (
            f"effective_rank(diag(3, 1, {tail:g}), {energy}) returned {got}, expected {want}: round each running "
            f"fraction to exactly 6 decimal places and compare it with the energy, not with a tolerance ({why})")
    assert effective_rank(np.zeros((4, 3)), 0.9) == 0, "an all-zero matrix has effective rank 0"
    for bad in ((np.zeros(3), 0.5), (np.zeros((2, 2, 2)), 0.5), (np.eye(2), 0.0), (np.eye(2), 1.01)):
        try:
            effective_rank(*bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"effective_rank with an argument of shape {np.shape(bad[0])} and energy "
                                 f"{bad[1]} must raise ValueError")
    print("exercise 9 looks right: energy is the squared singular value, counted from the largest down, "
          "and the rank of a rank-5 matrix is 5")

In [ ]:
if __name__ == "__main__":
    _try("exercise 9", _check_effective_rank)

In [ ]:
def _demo_spectrum() -> None:
    run = FULL[FULL_LR, SEEDS[0]][0]
    print("the update full fine-tuning made to each of the five matrices (learning rate "
          f"{FULL_LR:.0e}, seed {SEEDS[0]}):")
    print(f"{'matrix':<8}{'shape':>10}{'largest possible':>18}{'90% energy':>12}{'99% energy':>12}{'|dW| / |W|':>12}")
    ranks90 = []
    for name in LAYERS:
        delta = run.params[name] - BASE[name]
        r90, r99 = effective_rank(delta, 0.9), effective_rank(delta, 0.99)
        ranks90.append(r90)
        print(f"{name:<8}{str(BASE[name].shape):>10}{min(BASE[name].shape):>18}{r90:>12}{r99:>12}"
              f"{np.linalg.norm(delta) / np.linalg.norm(BASE[name]):>12.2f}")
    print(f"directions needed for 90% of the energy, over the five matrices: {min(ranks90)} to {max(ranks90)}")
    if KNEE_RUN is None:
        print("(the comparison with the adapter at the knee needs the rank sweep above)")
        return
    ad_ranks = [effective_rank(KNEE_RUN.scale * a["A"] @ a["B"], 0.99) for a in KNEE_RUN.adapters.values()]
    print(f"the rank-{KNEE} adapter's own update, at 99% energy: {min(ad_ranks)} to {max(ad_ranks)} directions "
          f"(it cannot exceed its rank, {KNEE})")
    below = sum(KNEE < r for r in ranks90)
    print(f"the knee rank, {KNEE}, is smaller than the full fine-tune's 90%-energy rank on {below} of the "
          f"{len(LAYERS)} matrices")

In [ ]:
if __name__ == "__main__":
    _try("the spectrum of the full update", _demo_spectrum, needs=("exercise 9",))

## 12. Where the adapters sit

The sweep put adapters on all five matrices. The original LoRA paper (its section 4.2) limited its study
to the attention weights and froze the MLP modules, and "LoRA Without Regret" later reported that where
adapters sit matters (sources in `claims.yaml`). This model
has no feed-forward block, so that finding cannot be reproduced here, but the question can be asked
of what the model does have: adapters on the four attention matrices only, on the read-out only, or
on all five, at one rank and one learning rate. One seed, so a difference smaller than the
seed-to-seed spread the sweep printed is not a finding.

In [ ]:
PLACEMENTS = (("attention (Wq Wk Wv Wo)", ("Wq", "Wk", "Wv", "Wo")),
              ("read-out (W1)", ("W1",)),
              ("all five", LAYERS))
PLACE_RANK, PLACE_LR = 8, 3e-2


def _demo_placement() -> None:
    rows = []
    for name, layers in PLACEMENTS:
        run = finetune(BASE, B_TRAIN, lr=PLACE_LR, seed=SEEDS[0], rank=PLACE_RANK, layers=layers)
        rows.append((name, run.trainable, score(run, B_VAL), score(run, B_TEST)))
    print(f"rank {PLACE_RANK}, learning rate {PLACE_LR:g}, seed {SEEDS[0]}, {STEPS} steps")
    print(f"{'adapters on':<26}{'trainable':>10}{'val':>8}{'test':>8}")
    for name, n, val, test in rows:
        print(f"{name:<26}{n:>10}{val:>8.3f}{test:>8.3f}")
    best = min(rows, key=lambda r: r[2])
    most = max(rows, key=lambda r: r[1])
    print(f"lowest validation loss: {best[0]}; most trainable parameters: {most[0]}; "
          f"{'the same placement' if best[0] == most[0] else 'not the same placement'}")

In [ ]:
if __name__ == "__main__":
    _try("where the adapters sit", _demo_placement, needs=_FOR_TRAINING)

## 13. A frozen base: merging, switching off, and forgetting

Three properties make an adapter useful beyond its size, and all three can be checked on the adapter the
sweep found at the knee:

1. **Merging is exact up to rounding.** Your `merge_adapter` folds the adapter into each matrix; the
   merged model, with no adapter at all, must give the same logits as the adapted model, and unmerging
   gets the base matrices back to rounding.
2. **The base never changed.** The sweep ran many fine-tunes from one `BASE`. If it is bit-for-bit what
   it was before them, one base can serve every adapter.
3. **Switching the adapter off restores the base exactly.** A full fine-tune here works on a copy of
   the weights, so switching between tasks needs a second full copy for each task; an adapter is a small
   add-on to one shared base. The cost of adapting shows up as held-out loss on domain A, the base's
   own domain. The paper "LoRA Learns Less and Forgets Less" (sources in `claims.yaml`) reports that
   LoRA forgets less than full fine-tuning at the scales it studied. The cell measures this model, and
   reports what it finds either way.

In [ ]:
def _demo_serving() -> None:
    run = KNEE_RUN
    probe = windows(B_VAL, np.random.default_rng(PROBE_SEED), 64)
    merged = dict(BASE)
    for name, adapter in run.adapters.items():
        merged[name] = merge_adapter(BASE[name], adapter, run.scale)
    with_adapter = forward(BASE, run.adapters, run.scale, probe[0])[0]
    without = forward(merged, {}, 1.0, probe[0])[0]
    print(f"1. the rank-{KNEE} adapter merged into five plain matrices: every logit within 1e-9 of the "
          f"adapted model's: {bool(np.abs(with_adapter - without).max() <= 1e-9)}")
    undone = all(np.abs(merge_adapter(merged[n], a, run.scale, unmerge=True) - BASE[n]).max() <= 1e-12
                 for n, a in run.adapters.items())
    print(f"   unmerged again, every matrix is within 1e-12 of the base's: {undone} "
          "(equal to rounding, which is why the lesson keeps the base itself rather than subtracting)")
    print("2. after every fine-tune of the sweep, BASE is bit-for-bit the checkpoint it started as: "
          f"{all(np.array_equal(BASE[k], BASE0[k]) for k in BASE)}")
    off = evaluate(BASE, {}, 1.0, A_TEST)
    print(f"3. adapter switched off, held-out A loss {off:.3f} nats: identical to before adapting: {off == _base_a}")
    lr = tuned_lr(SWEEP, KNEE)
    adapter_a = _col(SWEEP, KNEE, lr, 3)
    full_a = float(np.mean([FULL[FULL_LR, s][3] for s in SEEDS]))
    print(f"\nheld-out domain A, the base model's own world (nats per character):")
    print(f"  base model             {_base_a:7.3f}")
    print(f"  rank-{KNEE} adapter, on     {adapter_a:7.3f}   (mean of {len(SEEDS)} seeds; switch it off and this is the base again)")
    print(f"  full fine-tuned model  {full_a:7.3f}   (nothing to switch off)")
    print(f"  guessing uniformly     {np.log(VOCAB):7.3f}   (what a model that knows nothing about A scores)")
    print(f"after adapting to B, held-out A is {'worse' if adapter_a > full_a else 'better or equal'} with "
          f"the adapter on than after full fine-tuning on this run: {adapter_a:.3f} against {full_a:.3f}")
    if min(adapter_a, full_a) > np.log(VOCAB):
        print("both adapted models score worse on A than a uniform guess does: they were trained to put their "
              "probability on domain B's letters, so this number mostly measures that shift of the output "
              "distribution. It is not a clean measure of forgetting, and it cannot confirm or contradict the "
              "cited paper's finding at scale")

In [ ]:
if __name__ == "__main__":
    _try("a frozen base", _demo_serving, needs=("the rank sweep", "exercise 4"))

## 14. Self-check

1. A re-implementation initialises both `A` and `B` to zeros, "so that the adapter starts as a
   no-op". The training loss never moves from its starting value. Why?
   - (a) The learning rate is too small; zero weights need a larger one.
   - (b) A zero matrix has rank zero, so the adapter has no capacity.
   - (c) The gradient of each factor passes through the other, so both gradients are exactly zero and
         neither factor can ever leave zero.
   - (d) Zeros underflow in float64.

2. A team trains adapters of rank 1 to 64 at one fixed learning rate with the scale `alpha / r`. Held-out
   loss improves up to rank 16 and gets worse above it, and the team reports "this task cannot use more
   than rank 16". What is the most defensible reading?
   - (a) It is a statement about that learning rate as much as about the task: each rank can have its
         own best learning rate, which only a per-rank tune can show, and a knee needs repeating over
         seeds before it means anything.
   - (b) Ranks above 16 cannot be trained.
   - (c) The `alpha / r` rule is a bug that PEFT should have removed.
   - (d) The validation set is too small, and nothing else could explain it.

3. An adapter is merged into the base weights and the merged model is shipped. Which statement is true?
   - (a) The merged model has more parameters than the base, by the adapter's size.
   - (b) The merged model computes a different function from the adapted model and must be re-evaluated
         as a new model.
   - (c) Merging deletes the base weights, so the adapter cannot be recovered.
   - (d) The merged matrices have the base's shape, so serving needs no extra products; the function is
         the adapted model's, up to rounding, and undoing the merge needs the base weights or a subtraction.

4. Two matrices, 48 x 48 and 4096 x 4096, each get a rank-8 adapter. Which statement is correct?
   - (a) The adapter is the same fraction of the matrix in both cases.
   - (b) The adapter has about `2 * r / d` of the matrix's parameters, so it is a far smaller share of
         the large matrix than of the small one.
   - (c) The adapter is larger than the matrix in both cases.
   - (d) Adapters save memory only on small matrices.

5. A sweep, with a margin of 20 milli-nats, finds the knee at rank 8 from one seed. Two seeds, run
   afterwards, differ by 60 milli-nats in validation loss at rank 8. What follows?
   - (a) The knee is confirmed: the seeds agree to within the loss scale.
   - (b) The seeds are irrelevant when the loss is measured on validation data.
   - (c) The knee is not established: the noise between seeds is larger than the margin that defined it.
   - (d) The margin should be tightened until the seeds agree.

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
# Question 5, measured rather than taken on trust: how large is the seed-to-seed spread of the
# validation loss, rank by rank at each rank's chosen learning rate, against the margin that
# defines the knee?
def _demo_noise_against_margin() -> None:
    print(f"{'rank':>5}{'lr':>8}" + "".join(f"{'val, seed ' + str(s):>14}" for s in SEEDS)
          + f"{'spread, milli-nats':>20}")
    wider = []
    for rank in RANKS:
        lr = tuned_lr(SWEEP, rank)
        vals = [SWEEP[rank, lr, s][1] for s in SEEDS]
        spread = round(1000 * (max(vals) - min(vals)))
        if spread > MARGIN_MILLI:
            wider.append(rank)
        print(f"{rank:>5}{lr:>8.0e}" + "".join(f"{v:>14.3f}" for v in vals) + f"{spread:>20}")
    print(f"ranks whose two seeds differ by more than the {MARGIN_MILLI} milli-nat margin: "
          + (str(wider) if wider else "none"))

In [ ]:
if __name__ == "__main__":
    _try("seed spread against the margin", _demo_noise_against_margin, needs=("the rank sweep",))

## 15. Common mistakes

- **Initialising both factors to zero**, or `B` to something random. Zero-zero is dead; random `B` starts
  from a model that is not the one you are adapting. The reference initialisation has `B` at zero and `A`
  random.
- **Forgetting the scale in the backward pass.** The forward pass multiplies the adapter path by `s`, so both
  `dB` and the adapter part of `dx` carry it. Finite differences find this at once.
- **Computing `x @ (A @ B)` instead of `(x @ A) @ B`.** The result is the same and the point of LoRA is lost:
  the first builds a full-size matrix every call.
- **Training the base by accident.** A frozen matrix must get no gradient and no optimiser state. Check that the
  base is bit-for-bit unchanged after training, as section 13 does.
- **Reading a rank sweep at one learning rate.** The best rate for rank 4 need not be the best for rank 32
  (the scale rule changes the first steps with the rank). Tune it per rank on validation, report test.
- **Reporting a knee smaller than the noise.** Print the seed spread beside the margin.
- **Reading a rise in loss with rank as a capacity limit.** A larger adapter can represent every update a smaller
  one can: a worse loss at the larger rank comes from how it was trained, or from overfitting.
- **Calling a knee tuned when its best learning rate is the edge of the grid.** Say so when you report it.
- **Trusting a number that moves with the last bit.** If nudging every weight by one unit in the last place
  changes a result you report, the result belongs to the machine, not the model. Section 3 shows the test;
  a warm-up and a learning rate below the edge of stability are how it passes.
- **Writing `r * (d_in + d_out) <= d_in * d_out` for "fewer".** At the break-even rank the two counts are equal;
  an adapter that costs as much as the matrix saves nothing.
- **Merging in place.** If the base matrix is overwritten, the next adapter merges into the previous one's
  result.
- **Comparing losses by eye at a float edge.** Compare in whole milli-nats, as `find_knee` does.

## 16. What this lesson does NOT cover

- **Real models.** This model has one attention head and no feed-forward block, and its matrices are 48 wide.
  Findings about where adapters should sit in a deep model, how rank scales with dataset size, or how the best
  learning rate of LoRA compares with full fine-tuning's, are about models and data this lesson does not use.
  The measured numbers here describe this model, this vocabulary swap and this grid only.
- **QLoRA.** It keeps the frozen base in a 4-bit quantised format and trains LoRA adapters through it;
  quantisation is a separate lesson's subject, and nothing here quantises anything.
- **Other adapter methods and initialisations.** PEFT lists many (as of the date in `claims.yaml`, its
  `init_lora_weights` argument alone names several); each changes a part of what this lesson fixes.
- **Dropout and other regularisers on the adapter path**, which `peft.LoraConfig` also exposes.
- **Optional extension, not run by this lesson.** If you have PyTorch and PEFT installed, you can check your layer
  against the library's: build a plain linear layer, wrap it with a `LoraConfig` of the same rank and `lora_alpha`,
  copy your `A` and `B` into the adapter's weights, and compare outputs. Mind the layout: PEFT stores each
  matrix transposed relative to this lesson's `x @ W` convention. Names and layouts change between releases, so
  read the version you have before trusting a mismatch. This lesson does not depend on it and does not run it.

In [ ]:
# The closing recap. Every figure is recomputed here, not quoted from the prose above.
if __name__ == "__main__":
    print("--- what you built, measured ---")
    print(f"  parameters in the base model       {count_params(BASE)}")
    print(f"  a full fine-tune trains            {FULL[FULL_LR, SEEDS[0]][0].trainable}")
    if SWEEP is None:
        print("  the rank sweep                     not run yet: it needs exercises 1, 2, 3, 5 and 8")
    else:
        print(f"  adapter fine-tunes in the sweep    {len(SWEEP)}")
        print(f"  the knee, at {MARGIN_MILLI} milli-nats        rank {KNEE}, "
              f"{KNEE_RUN.trainable} trainable parameters")
    print("  every number above was computed by the code you just ran")

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
# The nine exercises it lists are the ones `_EXERCISES` names in the setup cell.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, function, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)"
                if status == "not started" and waiting not in (None, "", function) else "")
        print(f"  {_MARKS[status]}  {label} · {function}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, function, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]} — `{next_up[1]}`, {next_up[2]}. Its cell has hints if you "
              "are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check
    that ran and came back wrong is. In a script or under CI that ends the run non-zero, so a
    green exit code can never paper over it. In a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake "
              "where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.